In [123]:
import time
import psutil
import os
import threading
import traceback
import random
import copy
from pynvml import *
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional


In [124]:
import numpy as np
import cmath
import math

class GateType(Enum):
    X = auto()
    Y = auto()
    Z = auto()
    H = auto()
    S = auto()
    SDG = auto()
    T = auto()
    TDG = auto()
    SX = auto()
    V = auto()
    V_plus = auto()
    ARBITRARY = auto()

GATE_UNITARIES: Dict[GateType, List[List[complex]]]= {
    GateType.X: [[0, 1],
                [1, 0]],
    GateType.Y: [[0, -1j],
                [1j, 0]],
    GateType.Z: [[1, 0],
                [0, -1]],
    GateType.H: [[1/2**0.5, 1/2**0.5],
                [1/2**0.5, -1/2**0.5]],
    GateType.S:   [[1, 0],
                   [0,  1j]],
    GateType.SDG: [[1, 0],
                   [0, -1j]],
    GateType.T:   [[1, 0],
                   [0,  cmath.exp(1j*math.pi/4)]],
    GateType.TDG: [[1, 0],
                   [0,  cmath.exp(-1j*math.pi/4)]],
    GateType.SX:  [[0.5+0.5j, 0.5-0.5j],
                   [0.5-0.5j, 0.5+0.5j]],
    GateType.SXDG:[[0.5-0.5j, 0.5+0.5j],
                   [0.5+0.5j, 0.5-0.5j]],
    GateType.V:   [[0.5+0.5j, 0.5-0.5j],
                  [0.5-0.5j, 0.5+0.5j]],
    GateType.V_plus:  [[0.5-0.5j, 0.5+0.5j],
                  [0.5+0.5j, 0.5-0.5j]],
    GateType.ARBITRARY: [[1, 0],
                         [0, 1]]  # Identity as a placeholder for arbitrary gates
}

@dataclass
class SingleQubitGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int

@dataclass
class ControlledGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int
    control: List[int]

GateOperation = SingleQubitGate | ControlledGate


def operation_constructor(gate: GateType,
                          target: int,
                          control: Optional[List[int]] = None,
                          mode: Optional[str] = None,
                          matrix: Optional[List[List[complex]]] = None,
                          angles: Optional[List[float]] = None
) -> GateOperation:

    """
    Construct a quantum gate operation, either from a built-in gate type or an arbitrary unitary.

    This function returns either a SingleQubitGate or a ControlledGate depending on whether
    control qubits are specified. For GateType.ARBITRARY, you must specify `mode` as either
    'angles' (to generate the U3 matrix from Euler angles) or 'matrix' (to supply your own
    2x2 unitary).

    Args:
        gate (GateType):
            The gate type to construct. Use GateType.ARBITRARY for custom unitaries.
        target (int):
            Index of the target qubit.
        control (Optional[List[int]]):
            List of control-qubit indices. If None, a single-qubit gate is returned.
        mode (Optional[str]):
            Only for GateType.ARBITRARY.
            - 'angles': compute the unitary from `angles`.
            - 'matrix': validate and use the provided `matrix`.
        matrix (Optional[List[List[complex]]]):
            A 2x2 complex matrix for ARBITRARY gates when `mode='matrix'`.
        angles (Optional[List[float]]):
            Three Euler angles [theta, phi, lambda] for ARBITRARY gates when `mode='angles'`.

    Returns:
        GateOperation:
            A SingleQubitGate or ControlledGate instance encapsulating the specified operation.

    Raises:
        ValueError:
            - If `mode` is missing or invalid for an ARBITRARY gate.
            - If `angles` is not a list of exactly three floats when `mode='angles'`.
            - If `matrix` is missing, not 2x2, or not unitary when `mode='matrix'`.
    """

    def is_unitary(matrix: List[List[complex]], tol=1e-8) -> bool:
        M = np.array(matrix, dtype=complex)
        return np.allclose(M.conj().T @ M, np.eye(2), atol=tol)

    def check_matrix(matrix: List[List[complex]]):
        if matrix is None:
            raise ValueError("ARBITRARY gate requires a 2x2 matrix.")
        if len(matrix) != 2 or len(matrix[0]) != 2 or len(matrix[1]) != 2:
            raise ValueError("Matrix must be 2x2 for arbitrary gate.")
        if not is_unitary(matrix):
            raise ValueError("Provided matrix is not unitary.")

    def handle_angles(angles: List[float]) -> List[List[complex]]:
        if angles is None or len(angles) != 3:
            raise ValueError("Angles must be provided as a list of three values (theta, phi, lambda).")
        theta, phi, lam = angles
        a = np.cos(theta / 2)
        b = -np.exp(1j * lam) * np.sin(theta / 2)
        c = np.exp(1j * phi) * np.sin(theta / 2)
        d = np.exp(1j * (phi + lam)) * np.cos(theta / 2)
        matrix = [[a, b], [c, d]]
        return matrix

    if gate is GateType.ARBITRARY:
        if mode is None:
            raise ValueError("Mode must be specified for ARBITRARY gate.")
        if mode == 'angles':
            matrix = handle_angles(angles)
        elif mode == 'matrix':
            check_matrix(matrix)
        else:
            raise ValueError("Invalid mode for ARBITRARY gate. Use 'angles' or 'matrix'.")
    else:
        matrix = GATE_UNITARIES[gate]

    a, b = matrix[0]
    c, d = matrix[1]

    if control is None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)


def randomizer(option: Optional[GateOperation], threshold=0) -> List[GateOperation]:
    operations: List[GateOperation] = []

    for _ in range(NUM_OPERATIONS):
        gate = random.choice(list(GateType))
        matrix = GATE_UNITARIES[gate]
        a, b = matrix[0]
        c, d = matrix[1]

        if option == SingleQubitGate or random.random() < threshold:
            target = random.randint(0, NUM_QUBITS - 1)
            operations.append(SingleQubitGate(gate, a, b, c, d, target))
        elif option == ControlledGate or random.random() >= threshold:
            control, target = random.sample(range(NUM_QUBITS), 2)
            operations.append(ControlledGate(gate, a, b, c, d, target, [control]))

    return operations

def initialization(L, R) -> Tuple[List[List[complex]], List[List[complex]], List[List[int]]]:
    if 1 < INIT_MATRIX_SIZE:
        L[1, 0] = 1

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            L[row, col] = 1

    if 2 < INIT_MATRIX_SIZE:
        R[2, 0] = 1

    for col in range(1, INIT_MATRIX_SIZE - 2):
        if col % 2 == 1:
            row = col + 3
        else:
            row = col + 2
        if row < INIT_MATRIX_SIZE:
            R[row, col] = 1

    Mapping = []
    Mapping.append([0])

    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*i + 1])
    Mapping.append([2*i + 2])

    return L, R, Mapping

def initialization_sparse(INIT_MATRIX_SIZE, NUM_QUBITS, TYPE):
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse

    row_L, col_L, data_L = [], [], []
    row_R, col_R, data_R = [], [], []

    if 1 < INIT_MATRIX_SIZE:
        row_L.append(1)
        col_L.append(0)
        data_L.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            row_L.append(row)
            col_L.append(col)
            data_L.append(1)

    if 2 < INIT_MATRIX_SIZE:
        row_R.append(2)
        col_R.append(0)
        data_R.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 3 if col % 2 == 1 else col + 2
        if row < INIT_MATRIX_SIZE:
            row_R.append(row)
            col_R.append(col)
            data_R.append(1)

    L = sparse.csr_matrix((cp.array(data_L, dtype=cp.complex128), (cp.array(row_L), cp.array(col_L))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    R = sparse.csr_matrix((cp.array(data_R, dtype=cp.complex128), (cp.array(row_R), cp.array(col_R))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    def ones_positions_k(k: int):
        # [0], then pairs (1,2), (3,4), ..., (2k-3, 2k-2), then singles [2k-1], [2k]
        pos = [[0]]
        pos += [[i, i+1] for i in range(1, 2*k-1, 2)]  # k-1 pairs
        pos += [[2*k-1], [2*k]]
        return pos

    def wide_diag_matrix(k: int) -> sparse.csr_matrix:
        positions = ones_positions_k(k)
        rows = len(positions)          # k + 2
        cols = 2*k + 1                 # columns 0..2k

        # Build CSR arrays directly (indptr/indices/data)
        indptr = [0]
        indices = []
        data = []
        for idxs in positions:
            indices.extend(idxs)
            data.extend([1]*len(idxs))
            indptr.append(len(indices))

        indptr = cp.array(indptr, dtype=cp.int32)
        indices = cp.array(indices, dtype=cp.int32)
        data = cp.ones_like(indices, dtype=cp.complex128)

        return sparse.csr_matrix((data, indices, indptr), shape=(rows, cols))
    # Mapping = [[0]]
    # for i in range(1, NUM_QUBITS):
    #     Mapping.append([2*i - 1, 2*i])
    # Mapping.append([2*NUM_QUBITS - 1])
    # Mapping.append([2*NUM_QUBITS])

    Mapping = wide_diag_matrix(NUM_QUBITS)

    return L, R, Mapping
def _view_operations(operations: List[GateOperation]):
    for op in operations:
        print(op.gate.name, op.target)

In [125]:
### Fixed QASM parser with proper imports
import math, cmath, re
from pathlib import Path
from typing import Tuple, List

_PI_ENV = {"pi": math.pi, "tau": 2*math.pi, "e": math.e}

def _eval_expr(expr: str) -> float:
    """安全地把 'pi/2', '3*pi/4', '1.570796'… 轉成 float"""
    return eval(expr, {"__builtins__": None}, _PI_ENV)

def _rot_x(theta: float):
    c, s = math.cos(theta/2), math.sin(theta/2)
    return [[c, -1j*s], [-1j*s, c]]

def _rot_y(theta: float):
    c, s = math.cos(theta/2), math.sin(theta/2)
    return [[c, -s], [s, c]]

def _rot_z(theta: float):
    return [[cmath.exp(-1j*theta/2), 0],
            [0, cmath.exp(1j*theta/2)]]

def _phase(lambda_):
    return [[1, 0], [0, cmath.exp(1j*lambda_)]]

def extract_qubit_index(s: str) -> int:
    """Extract qubit index from string like 'q[0]' -> 0"""
    match = re.search(r'\[(\d+)\]', s)
    if match:
        return int(match.group(1))
    raise ValueError(f"Invalid qubit format: {s}")
## .reals file parser for revlib
def parse_reals(path: Path) -> Tuple[int, List[GateOperation]]:
    gate_str_to_enum = { 'v' : GateType.V, 'v+': GateType.V_plus}
    num_qubits, gate_list = None, []
    with open(path, 'r') as f:
        for raw in f:
            line = raw.strip().rstrip('\n')
            if not line or line.startswith('#'):
                continue
            # version ,varialbe, inputs, outputs, constants, garbage line (ignore)
            if line.startswith('.version') or line.startswith('.variable') or line.startswith('.inputs') or line.startswith('.outputs') or line.startswith('.constants') or line.startswith('.garbage') or line.startswith('.begin') or line.startswith('.end'):
                continue
            # numvars line
            if line.startswith('.numvars'):
                m = re.search(r'\d+', line)
                if m:
                    num_qubits = int(m.group(0))
                continue
            # gate line
            m = re.match(r'([a-zA-Z_][\w]*)(\([^)]*\))?\s+(.*)', line.strip())
            if not m:
                print(f"Malformed line: {line}"); continue
            token, para, qubits_part = m.groups()
            qubit_strs = [q.strip() for q in qubits_part.split()]
            ## deal with v v+ gate
            if token in {'v', 'v+'}:
                if len(qubit_strs) != 2:
                    print(f"Control/target mismatch in: {line}"); continue
                target = extract_qubit_index(qubit_strs[1])
                ctrls = [qubit_strs[0]]
                gate_enum = gate_str_to_enum[token]
                gate_op = operation_constructor(gate_enum, target, ctrls)
                gate_list.append(gate_op)
                continue
            if token.startswith('t'):
                if len(qubit_strs) == 1:
                    target = extract_qubit_index(qubit_strs[-1])
                    gate_op = operation_constructor(GateType.X, target)
                else:
                    target = extract_qubit_index(qubit_strs[-1])
                    ctrls = [extract_qubit_index(q) for q in qubit_strs[:-1]]
                    gate_op = operation_constructor(GateType.X, target, ctrls)
                gate_list.append(gate_op)
                continue
            if token.startswith('f'):
                if len(qubit_strs) == 1:
                    print(f"Control/target mismatch in: {line}"); continue
                else:
                    target1 = extract_qubit_index(qubit_strs[-1])
                    target2 = extract_qubit_index(qubit_strs[-2])
                    ctrls = [extract_qubit_index(q) for q in qubit_strs]
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target2, list(set(ctrls) - set([target2])),
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                    gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                    )
                continue
            if token.startswith('p'):
                if len(qubit_strs) != 3:
                    print(f"Control/target mismatch in: {line}"); continue
                else:
                    target1 = extract_qubit_index(qubit_strs[-2])
                    target2 = extract_qubit_index(qubit_strs[-1])
                    ctrls1 = [extract_qubit_index(q) for q in qubit_strs[:-2]]
                    ctrls2 = [extract_qubit_index(q) for q in qubit_strs[:-1]]
                    if token == 'pi' :
                        gate_list.append(
                            operation_constructor(GateType.X, target1, ctrls1)
                        )
                        gate_list.append(
                            operation_constructor(GateType.X, target2, ctrls2)
                        )
                        continue
                    else :
                        gate_list.append(
                            operation_constructor(GateType.X, target2, ctrls2)
                        )
                        gate_list.append(
                            operation_constructor(GateType.X, target1, ctrls1)
                        )
                        continue
    if num_qubits is None:
        raise ValueError("No .numvars line found in the file.")
    return num_qubits, gate_list


def parse_qasm(path: Path) -> Tuple[int, List[GateOperation]]:
    # 固定閘對應表
    gate_str_to_enum = {
        'x': GateType.X,   'y': GateType.Y,    'z': GateType.Z,  'h': GateType.H,
        's': GateType.S,   'sdg': GateType.SDG,
        't': GateType.T,   'tdg': GateType.TDG,
        'sx': GateType.SX, 'sxdg': GateType.SXDG,
    }

    num_qubits, gate_list = 0, []

    with open(path, 'r') as f:
        for raw in f:
            line = raw.strip().rstrip(';')
            if (not line or line.startswith('//')
                    or line.startswith('OPENQASM')
                    or line.startswith('include')):
                continue

            # qreg 宣告
            if line.startswith('qreg'):
                m = re.search(r'qreg\s+\w+\[(\d+)\]', line)
                if m:
                    num_qubits = num_qubits + int(m.group(1))
                continue
            # creg 宣告（忽略）
            if line.startswith('creg'):
                continue

            # barrier 宣告（忽略）
            if line.startswith('barrier'):
                continue
            # measure 宣告（忽略）
            if line.startswith('measure'):
                continue
            # ===== 解析 gate token、參數、qubit 清單 =====
            m = re.match(r'([a-zA-Z_][\w]*)(\([^)]*\))?\s+(.*)', line)
            if not m:
                print(f"Malformed line: {line}"); continue
            token, param_group, qubits_part = m.groups()
            qubit_strs = [q.strip() for q in qubits_part.split(',')]

            # ── 拆出控制字首 (c、cc …) ──
            ctrl_prefix = re.match(r'c+', token)
            num_ctrl = len(ctrl_prefix.group()) if ctrl_prefix else 0
            base_token = token[num_ctrl:]

            #處理swap gate (拆成3個cnot)
            if base_token == 'swap' or base_token == 'cswap':
                if len(qubit_strs) != 2 + num_ctrl:
                    print(f"Control/target mismatch in: {line}"); continue
                ctrls = [extract_qubit_index(q) for q in qubit_strs]
                target1 = extract_qubit_index(qubit_strs[-2])
                target2 = extract_qubit_index(qubit_strs[-1])
                print(ctrls, target1, target2)
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target2, list(set(ctrls) - set([target2])),
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target1, ctrls[-1],
                                          mode='matrix', matrix=[[0, 1], [1, 0]])
                )
                continue

            # ── 取得角度參數（若有） ──
            params = []
            if param_group:                       # "(theta,phi,…)" → list[str]
                params = [p.strip() for p in param_group[1:-1].split(',')]
                params = [_eval_expr(p) for p in params]

            # ── 驗證 qubit 數 ──
            if len(qubit_strs) != (num_ctrl + 1):
                print(f"Control/target mismatch in: {line}"); continue
            ctrls = [extract_qubit_index(q) for q in qubit_strs[:-1]]
            target = extract_qubit_index(qubit_strs[-1])

            # ===== 對應到 GateOperation =====
            #
            # 1. 固定單量子閘
            if not params and base_token in gate_str_to_enum:
                gate_enum = gate_str_to_enum[base_token]
                gate_op = operation_constructor(gate_enum, target,
                                                ctrls if num_ctrl else None)
                gate_list.append(gate_op); continue

            # 2. 參數化單量子閘（rx/ry/rz/p/u1/u2/u3）
            if base_token in {'rx','ry','rz','p','phase','u1','u2','u3'}:
                # 生成對應矩陣
                if base_token == 'rx': M = _rot_x(params[0])
                elif base_token == 'ry': M = _rot_y(params[0])
                elif base_token == 'rz': M = _rot_z(params[0])
                elif base_token in ['p','phase','u1']: M = _phase(params[0])
                elif base_token == 'u2':
                    phi, lam = params
                    # U2 gate: RZ(phi) · RY(π/2) · RZ(lam)
                    c, s = math.cos(math.pi/4), math.sin(math.pi/4)
                    M = [[c, -s*cmath.exp(1j*lam)],
                         [s*cmath.exp(1j*phi), c*cmath.exp(1j*(phi+lam))]]
                elif base_token == 'u3':
                    theta, phi, lam = params
                    # U3 gate: general single-qubit unitary
                    c, s = math.cos(theta/2), math.sin(theta/2)
                    M = [[c, -s*cmath.exp(1j*lam)],
                         [s*cmath.exp(1j*phi), c*cmath.exp(1j*(phi+lam))]]

                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target,
                                          ctrls if num_ctrl else None,
                                          mode='matrix', matrix=M)
                )
                continue

            # 3. 控制版（cp/crx/cry/crz）
            if base_token in {'p','rx','ry','rz'} and num_ctrl >= 1:
                # 重用上面矩陣生成功能
                if base_token == 'p': M = _phase(params[0])
                elif base_token == 'rx': M = _rot_x(params[0])
                elif base_token == 'ry': M = _rot_y(params[0])
                elif base_token == 'rz': M = _rot_z(params[0])
                gate_list.append(
                    operation_constructor(GateType.ARBITRARY, target,
                                          ctrls, mode='matrix', matrix=M)
                )
                continue

            print(f"Skipping unsupported or malformed line: {line}")

    if num_qubits is None:
        raise ValueError("No qreg declaration found in QASM file.")

    return num_qubits, gate_list

def parse_qasm_to_custom(path: Path): ##general parser
    if path.suffix == '.qasm':
        return parse_qasm(path)
    elif path.suffix == '.real':
        return parse_reals(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")


In [126]:
NUM_QUBITS = 4
# NUM_OPERATIONS = 20
INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

# operations = randomizer(option=ControlledGate)

operations : List[GateOperation] = []
operations.append(operation_constructor(GateType.H, 1))
operations.append(operation_constructor(GateType.H, 2))
# operations.append(operation_constructor(GateType.H, 0))
# operations.append(operation_constructor(GateType.H, 1))
operations.append(operation_constructor(GateType.X, 3, [1,2]))
# operations.append(operation_constructor(GateType.X, 3, [1,2]))
# operations.append(operation_constructor(GateType.H, 0))
# operations.append(operation_constructor(GateType.H, 1))
# operations.append(operation_constructor(GateType.H, 0, [1, 2]))
# operations.append(operation_constructor(GateType.X, 2, [1]))

# operations.append(operation_constructor(GateType.Z, 2))

# _view_operations(operations)

In [127]:
def compute_result(L, R, seq: List[int], leaf: List[int], TYPE) -> complex:
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse

    A = cp.zeros((1, L.shape[0]))

    A[0, leaf] = 1

    B = cp.zeros((L.shape[0], 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs

    result = A @ dirs

    return result.item()

class SimulationAllPath:
    def __init__(self, L, R, leaf, NUM_QUBITS, TYPE):
        self.L = L
        self.R = R
        self.LL = self.L@self.L
        self.RR = self.R@self.R
        self.LR = self.L@self.R
        self.RL = self.R@self.L
        self.leaf = leaf
        self.NUM_QUBITS = NUM_QUBITS
        self.TYPE = TYPE
        self.sim_result = {}
        self.sim_time = 0

    def recursion(self, seq: List[int], dirs, A, start_time):
        if time.time() - start_time > 60:
            raise TimeoutError("Simulation Timeout: 60s.")
        if len(seq) == self.NUM_QUBITS:
            result = A @ dirs
            val = result.toarray()[0, 0]
            if abs(val) > 1e-10:
                self.sim_result[tuple(seq)] = val
            return

        if self.NUM_QUBITS <= 10 or len(seq) < 2:
            for dir in [0, 1]:
                next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                self.recursion(seq + [dir], next_dirs, A, start_time)
        else:
            for d1 in [0, 1]:
                for d2 in [0, 1]:
                    if d1 == 0 and d2 == 0:
                        next_dirs = self.LL @ dirs
                    elif d1 == 0 and d2 == 1:
                        next_dirs = self.RL @ dirs
                    elif d1 == 1 and d2 == 0:
                        next_dirs = self.LR @ dirs
                    elif d1 == 1 and d2 == 1:
                        next_dirs = self.RR @ dirs

                    self.recursion(seq + [d1, d2], next_dirs, A, start_time)

    def simulate_non_recursive(self, A, B, start_time):
        stack = [([], B)]  # (sequence_so_far, current_dirs)

        while stack:
            if time.time() - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            if self.NUM_QUBITS <= 10 or len(seq) == self.NUM_QUBITS - 1:
                for dir in [0, 1]:
                    next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                    stack.append((seq + [dir], next_dirs))
            else:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        if d1 == 0 and d2 == 0:
                            next_dirs = self.LL @ dirs
                        elif d1 == 0 and d2 == 1:
                            next_dirs = self.RL @ dirs
                        elif d1 == 1 and d2 == 0:
                            next_dirs = self.LR @ dirs
                        else:  # d1 == 1 and d2 == 1
                            next_dirs = self.RR @ dirs

                        stack.append((seq + [d1, d2], next_dirs))

    def build_4bit_matrix_map(self):
        from itertools import product

        mat_map = {}
        for bits in product([0, 1], repeat=4):
            mat = self.L if bits[0] == 0 else self.R
            for b in bits[1:]:
                mat = (self.L if b == 0 else self.R) @ mat
            mat_map[bits] = mat
        return mat_map

    def simulate_non_recursive_4bit(self, A, B, start_time):
        matrix4_map = self.build_4bit_matrix_map()
        stack = [([], B)]

        while stack:
            now_time = time.time()
            if now_time - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            remaining = self.NUM_QUBITS - len(seq)
            if remaining >= 4:
                for bits4, mat in matrix4_map.items():
                    next_seq = seq + list(bits4)
                    next_dirs = mat @ dirs
                    stack.append((next_seq, next_dirs))
            elif remaining >= 2:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        next_dirs = (
                            self.LL @ dirs if d1 == 0 and d2 == 0 else
                            self.RL @ dirs if d1 == 0 and d2 == 1 else
                            self.LR @ dirs if d1 == 1 and d2 == 0 else
                            self.RR @ dirs
                        )
                        stack.append((seq + [d1, d2], next_dirs))
            else:
                for d in [0, 1]:
                    next_dirs = self.L @ dirs if d == 0 else self.R @ dirs
                    stack.append((seq + [d], next_dirs))

            # print(time.time() - now_time, "seconds elapsed for current iteration:", seq)


    def simulation(self):
        if self.TYPE == 'numpy':
            import numpy as cp
            from scipy import sparse
        elif self.TYPE == 'cupy':
            import cupy as cp
            from cupyx.scipy import sparse

        A = cp.zeros((1, self.L.shape[0]))
        A[0, self.leaf] = 1
        A = sparse.csr_matrix(A, dtype=complex)

        B = cp.zeros((self.L.shape[0], 1))
        B[0, 0] = 1
        B = sparse.csr_matrix(B, dtype=complex)

        start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)
        self.simulate_non_recursive_4bit(A, B, start_simulation)
        self.sim_time = time.time() - start_simulation


    def __iter__(self):
        ...
        # A = cp.zeros((1, self.L.shape[0]))
        # A[0, self.leaf] = 1
        # A = sparse.csc_matrix(A, dtype=cp.complex128)

        # B = cp.zeros((self.L.shape[0], 1))
        # B[0, 0] = 1
        # B = sparse.csc_matrix(B, dtype=cp.complex128)

        # start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)

        # for i in range(2**self.NUM_QUBITS):
        #     if time.time() - start_simulation > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")

        #     seq = [(i >> j) & 1 for j in reversed(range(self.NUM_QUBITS))]
        #     dirs = B
        #     if self.NUM_QUBITS <= 10:
        #         for dir in seq:
        #             dirs = self.L @ dirs if dir == 0 else self.R @ dirs
        #     else:
        #         for d1, d2 in zip(seq[::2], seq[1::2]):
        #             if d1 == 0 and d2 == 0:
        #                 dirs = self.LL @ dirs
        #             elif d1 == 0 and d2 == 1:
        #                 dirs = self.RL @ dirs
        #             elif d1 == 1 and d2 == 0:
        #                 dirs = self.LR @ dirs
        #             elif d1 == 1 and d2 == 1:
        #                 dirs = self.RR @ dirs

        #         if self.NUM_QUBITS % 2 == 1:
        #             d = seq[-1]
        #             dirs = self.L @ dirs if d == 0 else self.R @ dirs

        #     result = A @ dirs
            # val = result.toarray()[0, 0]
            # val = result.item()
            # print("L: ", self.L)
            # print("R: ", self.R)
            # print("Leaf: ", self.leaf)
            # print(f"Path: {seq}, Value: {val}")
            # if abs(val) > 1e-10:
            #     self.sim_result[tuple(seq)] = val
            # yield val, seq

    def results(self):
        return self.sim_result

    def simulation_time(self):
        return self.sim_time

def simulation_all_path(L, R, leaf, NUM_QUBITS, TYPE, START_TIME):
    sim = SimulationAllPath(L, R, leaf, NUM_QUBITS, TYPE)
    sim.simulation()
    # for val, seq in sim:
    #     pass  # Iterate through all paths to compute results
    return sim.results(), sim.simulation_time()



In [128]:
def show_matrix(matrix: List[List[complex]]):
    """
    Display a matrix in a readable format.
    """
    for row in matrix:
        print(" | ".join(f"{val:8.4f}" for val in row))
    print("\n")

def show_adj_list(adj_list: Dict[int, Dict[int, Tuple[float, float]]]):
    """
    Display the adjacency list in a readable format.

    Args:
        adj_list (Dict[int, Dict[int, Tuple[float, float]]]):
            The adjacency list to display.
    """
    for node, edges in adj_list.items():
        print(f"Node {node}:")
        for child, (l_val, r_val) in edges.items():
            print(f"-> Child {child}: L={l_val:.2f}, R={r_val:.2f}")

class WeakSimulation:

    root_index = 0

    def __init__(self,
                 L,
                 R,
                 Mapping: List[List[int]],
                 NUM_QUBITS: int,
                 TYPE: str = 'numpy',
                 START_TIME: float = 0.0,
                 num_shots: int = 1024,
                 tol: float = 1e-10,
                 DEBUG: bool = False
    ):
        self.L = L
        self.R = R
        self.Mapping = Mapping
        self.NUM_QUBITS = NUM_QUBITS
        self.TYPE = TYPE
        self.START_TIME = START_TIME
        self.num_shots = num_shots
        self.tol = tol
        self.DEBUG = DEBUG

        if self.TYPE == 'numpy':
            import numpy as cp
            import scipy.sparse as sparse

        elif self.TYPE == 'cupy':
            import cupy as cp
            import cupyx.scipy.sparse as sparse

        else:
            raise ValueError("Unsupported TYPE. Use 'numpy' or 'cupy'.")

        self.cp = cp
        self.sparse = sparse

        self.one_terminals = Mapping[-2]
        self.zero_terminals = Mapping[-1]
        self.num_of_nodes = L.shape[0]
        self.root_index = 0

        self.adj_list = {}
        self.parent_map = {}

        self.sim_result = {}
        self.probs = {}
        self.downstream = {}
        self.upstream = {}

        self.reachable_set = set()
        self.pruned_set = set()
        assert(self.NUM_QUBITS + 2 == len(self.Mapping))
        self.my_Mapping = [[] for _ in range(self.NUM_QUBITS + 2)]

    def build_adj_list(self):

        if self.DEBUG:
            print("[WEAK_SIM] Building adjacency list and parent map...")
            print("[WEAK_SIM] Number of nodes:", self.num_of_nodes)

        from collections import defaultdict

        L_coo = self.L.tocoo()
        R_coo = self.R.tocoo()

        edges = defaultdict(lambda: [0.0, 0.0])

        for j, i, v in zip(L_coo.row, L_coo.col, L_coo.data):
            edges[(int(i), int(j))][0] = float(abs(v) ** 2)

        for j, i, v in zip(R_coo.row, R_coo.col, R_coo.data):
            edges[(int(i), int(j))][1] = float(abs(v) ** 2)


        if self.DEBUG:
            print("[WEAK_SIM] Edges found:", len(edges))

        for (i, j), (wL, wR) in edges.items():

            if i in self.adj_list:
                self.adj_list[i][j] = (wL, wR)
            else:
                self.adj_list[i] = {j: (wL, wR)}

            if j not in self.adj_list:
                self.adj_list[j] = {}

    def verify_direction(self, Mapping):
        level = {}
        for lvl, nodes in enumerate(Mapping):
            for n in nodes:
                level[n] = lvl

        bad_edges = []
        for p, children in self.adj_list.items():
            for c in children:
                if level[p] >= level[c] and c not in Mapping[-2] and c not in Mapping[-1]:
                    bad_edges.append((p, c, level[p], level[c]))

        if len(bad_edges) > 0:
            print("[Error] Bad edges found in the adjacency list:")
            for p, c, lvl_p, lvl_c in bad_edges:
                print(f"Parent {p} (level {lvl_p}) -> Child {c} (level {lvl_c})")
            print("[Error] The adjacency list does not respect the mapping levels.")
        else:
            print("[INFO] All edges in the adjacency list respect the mapping levels.")

    def prune_edges(self, node_index: int = root_index) -> bool:

        if self.DEBUG:
            print(f"[WEAK_SIM] Pruning edges for node {node_index}")

        if node_index in self.zero_terminals:
            self.pruned_set.add(node_index)
            return True
        if node_index in self.one_terminals:
            return False

        all_child_should_be_pruned = True

        child_nodes = list(self.adj_list[node_index].keys())

        for child_node in child_nodes:
            if self.prune_edges(child_node):
                del self.adj_list[node_index][child_node]
            else:
                all_child_should_be_pruned = False

        if all_child_should_be_pruned:
            self.pruned_set.add(node_index)

        return all_child_should_be_pruned

    def build_parent_map(self):
        """
        Build a parent map for the adjacency list.
        This map will help in computing upstream contributions.
        """

        for parent_node, children in self.adj_list.items():
            for child_node in children:
                if child_node not in self.parent_map:
                    self.parent_map[child_node] = []
                self.parent_map[child_node].append(parent_node)

    def build_reachable_set(self, node_index: int = root_index):
        """
        Build a set of reachable nodes starting from the root node.
        This is used to ensure that we only compute contributions for reachable nodes.
        """

        if node_index in self.reachable_set:
            return

        self.reachable_set.add(node_index)

        for child_node in self.adj_list[node_index]:
            self.build_reachable_set(child_node)

    def check_loop(self):
        """
        Check if there are any loops in the adjacency list.
        This is used to ensure that the graph is acyclic.
        """

        visited = set()
        stack = set()

        def visit(node_index):
            if node_index in stack:
                print(f"[Error] Loop detected at node {node_index}")
                return True

            if node_index in visited:
                return False

            visited.add(node_index)
            stack.add(node_index)

            for child_node in self.adj_list.get(node_index, {}):
                if visit(child_node):
                    return True

            stack.remove(node_index)
            return False

        for node_index in self.adj_list:
            if visit(node_index):
                return True

        print("[INFO] No loops detected.")
        return False

    def build_my_mapping(self, node_index: int = root_index, level: int = 0):

        if node_index not in self.my_Mapping[level]:
            if node_index in self.Mapping[-1]:
                self.my_Mapping[-1].append(node_index)
            elif node_index in self.Mapping[-2]:
                self.my_Mapping[-2].append(node_index)
            else:
                self.my_Mapping[level].append(node_index)
        else:
            return


        if node_index in self.adj_list:
            for child_node in self.adj_list[node_index]:
                self.build_my_mapping(child_node, level + 1)

    def check_two_mappings_are_equal(self):
        """
        Check if the two mappings are equal.
        This is used to ensure that the mapping is consistent with the adjacency list.
        """
        are_equal = True

        for i in range(len(self.my_Mapping)):
            if set(self.my_Mapping[i]) != set(self.Mapping[i]):
                # print("[Error] Mappings are not equal at level", i)
                are_equal = False

        if are_equal:
            print("[INFO] Mappings are equal.")
        else:
            print("[Error] Mappings are not equal.")

    def check_mapping_and_parent_map(self, Mapping: List[List[int]]) -> bool:
        """
        Check if the mapping and parent map are consistent.
        This is used to ensure that the parent map is correctly built from the adjacency list.
        """

        consistent = True

        for level in range(1, len(Mapping) - 1):
            for node_index in Mapping[level]:
                if node_index not in self.parent_map:
                    print(f"[Error] Node {node_index} at level {level} is not in parent_map.")
                    consistent = False
                else:
                    for parent_node in self.parent_map[node_index]:
                        if parent_node not in Mapping[level - 1]:
                            print(f"[Error] Parent node {parent_node} of {node_index} is not in the previous level mapping.")
                            consistent = False

        for node_index in Mapping[-1]:
            if node_index not in self.parent_map:
                    print(f"[Error] Node {node_index} at level {level} is not in parent_map.")
                    consistent = False
            else:
                for parent_node in self.parent_map[node_index]:
                    if parent_node not in Mapping[-3]:
                        print(f"[Error] Parent node {parent_node} of {node_index} is not in the previous level mapping.")
                        consistent = False

        return consistent

    def filter_adj_list(self):
        """
        Filter the adjacency list to only include reachable nodes.
        This is used to optimize the simulation by ignoring unreachable nodes.
        """

        filtered_adj_list = {}
        for node_index in self.reachable_set:
            if node_index in self.adj_list:
                filtered_adj_list[node_index] = self.adj_list[node_index]

        self.adj_list = filtered_adj_list

    def filter_parent_map(self):
        """
        Filter the parent map to only include reachable nodes.
        This is used to optimize the upstream contributions computation.
        """

        filtered_parent_map = {}
        for node_index in self.reachable_set:
            if node_index in self.parent_map:
                filtered_parent_map[node_index] = list(set(self.parent_map[node_index]) & self.reachable_set)

        self.parent_map = filtered_parent_map

    def check_node(self, node_index: int):
        """
        Check if a node is reachable and exists in the adjacency list.
        If not, raise an error.
        """

        print(f"\n[Error] Checking node:, {node_index}")

        if node_index not in self.reachable_set:
            print(f"[Error] Node {node_index} is not reachable from the root node.")
        else:
            print(f"[Error] Node {node_index} is reachable from the root node.")

        if node_index in self.pruned_set:
            print(f"[Error] Node {node_index} has been pruned.")
        else:
            print(f"[Error] Node {node_index} hasn't been pruned.")

        if node_index not in self.adj_list:
            print(f"[Error] Node {node_index} does not exist in the adjacency list.")
        else:
            print(f"[Error] Children of node {node_index}: {list(self.adj_list[node_index].keys())}")

        if node_index not in self.parent_map:
            print(f"[Error] Node {node_index} is not in parent_map.")
        else:
            print(f"[Error] Node {node_index} is in parent_map.")

    def compute_downstream_contributions(self, node_index: int = root_index) -> float:

        if node_index in self.one_terminals:
            self.downstream[node_index] = 1.0
            return 1.0

        if node_index in self.zero_terminals:
            self.downstream[node_index] = 0.0
            return 0.0

        contributions = 0

        for child_node, (l_val, r_val) in self.adj_list[node_index].items():
            contributions += (l_val + r_val) * self.compute_downstream_contributions(child_node)

        self.downstream[node_index] = contributions

        return contributions

    def compute_upstream_contributions(self):

        from collections import deque

        Q = deque()
        Q.append(self.root_index)
        self.upstream[self.root_index] = 1.0

        miss_compute = False

        while Q:
            node_index = Q.popleft()

            if node_index not in self.one_terminals and node_index not in self.zero_terminals:
                for child_node in self.adj_list[node_index]:
                    Q.append(child_node)

            if node_index in self.upstream:
                continue

            contributions = 0

            for parent_node in self.parent_map[node_index]:
                if parent_node not in self.upstream:
                    # self.check_node(parent_node)
                    if not miss_compute:
                        print(f"[Error] Upstream of node {parent_node}, ", end="")
                        miss_compute = True
                    else:
                        print(f"{parent_node}, ", end="")
                    continue
                l_val, r_val = self.adj_list[parent_node][node_index]
                contributions += (l_val + r_val) * self.upstream[parent_node]

            self.upstream[node_index] = contributions

        if miss_compute:
            print("have not been computed")

    def compute_upstream_contributions_with_mapping(self, Mapping: List[List[int]]):
        """
        Compute upstream contributions using the Mapping.
        This is used to ensure that we correctly compute contributions for each node.
        """

        miss_compute = False

        for level in range(len(Mapping)):
            for node in Mapping[level]:
                if node == self.root_index:
                    self.upstream[node] = 1.0
                    continue

                self.upstream[node] = 0.0

                for parent_node in self.parent_map[node]:
                    if parent_node not in self.upstream:
                        if not miss_compute:
                            print(f"[Error] Upstream of node {parent_node}, ", end="")
                            miss_compute = True
                        else:
                            print(f"{parent_node}, ", end="")
                        continue

                    l_val, r_val = self.adj_list[parent_node][node]
                    self.upstream[node] += (l_val + r_val) * self.upstream[parent_node]

        if miss_compute:
            print("have not been computed")

    def simulation_once(self) -> str:

        result = ""

        processing_nodes = []
        processing_nodes.append(self.root_index)

        for level in range(self.NUM_QUBITS):

            zero_prob = 0.0

            for node in processing_nodes:
                for child in self.adj_list[node]:

                    if node not in self.reachable_set or child not in self.reachable_set:
                        raise ValueError(f"Node {node} or child {child} is not reachable from the root node.")

                    l_val = self.adj_list[node][child][0]
                    if l_val > 0:
                        zero_prob += l_val * self.upstream.get(node, 0) * self.downstream.get(child, 0)

            next_processing_nodes = []

            if random.random() < zero_prob:
                result += "0"
                for node in processing_nodes:
                    for child in self.adj_list[node]:
                        l_val = self.adj_list[node][child][0]
                        if l_val > self.tol:
                            next_processing_nodes.append(child)
            else:
                result += "1"
                for node in processing_nodes:
                    for child in self.adj_list[node]:
                        r_val = self.adj_list[node][child][1]
                        if r_val > self.tol:
                            next_processing_nodes.append(child)

            processing_nodes = next_processing_nodes

        return result

    def simulation(self):

        self.build_adj_list()

        # self.verify_direction(self.Mapping)

        if self.DEBUG:
            print("[WEAK_SIM] adj_list:")
            show_adj_list(self.adj_list)

        # if self.DEBUG: print("[WEAK_SIM] Pruning Edges...")

        # self.prune_edges()

        # if self.DEBUG: print("[WEAK_SIM] Edges after pruning:")
        # if self.DEBUG: show_adj_list(self.adj_list)

        self.build_parent_map()

        if self.DEBUG: print("[WEAK_SIM] Parent Map:", self.parent_map)

        self.build_reachable_set()

        # self.check_loop()

        if self.DEBUG: print("[WEAK_SIM] Reachable Set:", self.reachable_set)

        # self.build_my_mapping()

        # self.check_two_mappings_are_equal()

        # print("[WEAK_SIM] Checking Mapping and Parent Map consistency...")
        # if self.check_mapping_and_parent_map(self.Mapping):
        #     print("[Info] Mapping and Parent Map are consistent.")
        # else:
        #     print("[Error] Mapping and Parent Map are NOT consistent.")

        # print("[WEAK_SIM] Checking My Mapping and Parent Map consistency...")
        # if self.check_mapping_and_parent_map(self.my_Mapping):
        #     print("[Info] My Mapping and Parent Map are consistent.")
        # else:
        #     print("[Error] My Mapping and Parent Map are NOT consistent.")


        # self.filter_adj_list()
        # if self.DEBUG: print("[WEAK_SIM] Filtered Adjacency List:")
        # if self.DEBUG: show_adj_list(self.adj_list)

        # self.filter_parent_map()
        # if self.DEBUG: print("[WEAK_SIM] Filtered Parent Map:", self.parent_map)

        if self.DEBUG: print("[WEAK_SIM] Computing Downstream Contributions...")

        self.compute_downstream_contributions()

        if self.DEBUG: print("[WEAK_SIM] Downstream Contributions:", self.downstream)

        if self.DEBUG: print("[WEAK_SIM] Computing Upstream Contributions...")

        self.compute_upstream_contributions()

        # self.compute_upstream_contributions_with_mapping(self.my_Mapping)

        if self.DEBUG: print("[WEAK_SIM] Upstream Contributions:", self.upstream)

        if self.DEBUG: print("[WEAK_SIM] Starting Simulation...")

        for _ in range(self.num_shots):
            if time.time() - self.START_TIME > 60:
                raise TimeoutError("Timeout during simulation: 60s.")

            result = self.simulation_once()

            if result in self.sim_result:
                self.sim_result[result] += 1
            else:
                self.sim_result[result] = 1

        if self.DEBUG: print("[WEAK_SIM] Simulation Results:", self.sim_result)

    def get_result(self) -> Dict[str, int]:

        if not self.sim_result:
            self.simulation()
        return self.sim_result

    def compute_probability(self):

        for key, value in self.sim_result.items():
            self.probs[key] = value / self.num_shots
        return self.probs

    def get_probs(self) -> Dict[str, float]:

        if not self.probs:
            if not self.sim_result:
                self.simulation()
            self.compute_probability()
        return self.probs

def weak_simulation(L, R, Mapping: List[List[int]], NUM_QUBITS: int, TYPE: str = 'numpy', START_TIME: float = 0.0, num_shots: int = 1024, tol: float = 1e-10):
    start_time = time.time()
    weak_sim = WeakSimulation(L, R, Mapping, NUM_QUBITS, TYPE, time.time(), num_shots, tol)
    return weak_sim.get_result(), time.time() - start_time


In [129]:
def dd_sparse_old(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int, TYPE: str, START_TIME: float, MODE: str = 'strong_simulation', DEBUG = False):
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS, TYPE)
    # print("Init Mapping: ", Mapping)
    # print("L:", L)
    # print("R:", R)

    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse

    for op in operations:
        if time.time() - START_TIME > 60:
            raise TimeoutError("Timeout in Circuit Operations: 60s.")

        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)

        target_map = Mapping.getrow(op.target).indices.tolist()

        A = cp.ones(MATRIX_SIZE, dtype=complex)
        A[target_map] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=complex)
        B[target_map] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=complex)
        C[target_map] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=complex)
        D[target_map] = op.d

        A = sparse.diags(A, offsets=0, format='csr')
        B = sparse.diags(B, offsets=0, format='csr')
        C = sparse.diags(C, offsets=0, format='csr')
        D = sparse.diags(D, offsets=0, format='csr')


        L_U = L @ A + R @ B
        R_U = L @ C + R @ D
        Mapping_U = Mapping.copy()

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)

        elif isinstance(op, ControlledGate):
            MATRIX_SIZE_OLD = MATRIX_SIZE
            control_up = [control for control in op.control if control < op.target]
            control_down = [control for control in op.control if control > op.target]

            # if len(control_up) != 0:
            # from itertools import chain
            # node_up = list(chain.from_iterable(Mapping.getrow(i).indices.tolist() for i in control_up))
            #     # L_up =


            # if len(control_down) != 0:


            for control in op.control:
                keep_U = cp.ones(MATRIX_SIZE, dtype=int)
                keep = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                map_control = Mapping.getrow(control).indices.tolist()
                keep_U[map_control] = 0
                keep[map_control] = 0
                M_U = sparse.diags(keep_U, offsets=0, format='csr')
                M = sparse.diags(keep, offsets=0, format='csr')

                # print("M_U: ", M_U)
                # print("M: ", M)
                # print("L: ", L)
                # print("L[1:, 0]: ", L[1:, 0].shape)
                # print("L[1:, 1:]: ", L[1:, 1:].shape)
                # print("L_U @ M_U: ", (L_U @ M_U).shape)

                L_U_old = L_U @ M_U

                L_U = sparse.bmat([[L_U_old[:, 0:1], L_U_old[:, 1:], None, None],
                                   [L[:, 0:1], None, None, L[:, :]]], format='csr')

                R_old = R @ M
                # print("R_old: ", R_old)
                # print("R_old[1:, 0]: ", R_old[1:, 0:1])
                # print("R_old[1:, 1:]: ", R_old[1:, 1:])

                R_U = sparse.bmat([[R_U[:, 0:1], R_U[:, 1:], None, None],
                                   [R_old[:, 0:1], None, None, R_old[:, :]]], format='csr')

                # if control < op.target:
                #     R_masked = R @ M
                #     R_U_masked = R_U[:, :MATRIX_SIZE_OLD] @ M_U

                #     L_U = sparse.bmat([[L, None],
                #         [None, L_U]], format='csr')

                #     # print("Mapping: ", Mapping[control])

                #     # print("D: ", D)
                #     # print("D_complement: ", D_complement)
                #     # print("R: ", R)
                #     # print("R_U: ", R_U)
                #     # print("R_U[]: ", R_U[:, :MATRIX_SIZE_OLD])
                #     # print("R_masked: ", R_masked)
                #     # print("R_U_masked: ", R_U_masked)


                #     # if TYPE == 'cupy':
                #     #     print("R Masked Type: ", type(R_masked))
                #     #     print("R U Masked Type: ", type(R_U_masked))
                #     #     print("R U Type: ", type(R_U))
                #     #     print("R Masked shape: ", R_masked.shape)
                #     #     print("R U Masked shape: ", R_U_masked.shape)
                #     #     print("R_U shape: ", R_U.shape)
                #         # print("R_masked max row/col:", R_masked.nonzero())
                #         # print("R_U_masked max row/col:", R_U_masked.nonzero())
                #         # print("R_U max row/col:", R_U.nonzero())

                #     R_U = sparse.bmat([[R_masked, None],
                #         [R_U_masked, R_U]], format='csr')

                #     # print("L_U: ", L_U)
                #     # print("R_U: ", R_U)
                # else:

                #     L_U_masked = L_U @ D_complement
                #     L_masked = L[:, :MATRIX_SIZE_OLD] @ D
                #     L_U = sparse.bmat([[L_U_masked, None],
                #         [L_masked, L]], format='csr')

                #     R_U = sparse.bmat([[R_U, None],
                #         [None, R]], format='csr')

                MATRIX_SIZE += MATRIX_SIZE_OLD
                Mapping_U = sparse.hstack([Mapping_U, Mapping], format='csr')

            L, R, Mapping = L_U, R_U, Mapping_U
            # print("L: ", L)
            # print("R: ", R)
            # print("start traverse")
            if DEBUG:
                print("L: ", L)
                print("R: ", R)
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)

            traverse = sparse.csr_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)

            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            # Mapping_U = [
            #     sorted(
            #         [n + MATRIX_SIZE_OLD * i
            #         for i in range(1, int(MATRIX_SIZE / MATRIX_SIZE_OLD) + 1)
            #         for n in mapping
            #         if n + MATRIX_SIZE_OLD * i in traverse] +
            #         [n for n in mapping if n in traverse]
            #     )
            #     for mapping in Mapping
            # ]
            # print("Mapping_U: ", Mapping_U)

            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            # Mapping = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U after dict: ", Mapping_U)
            # Mapping = copy.deepcopy(Mapping_U)
            # print("end mapping, time spent:", time.time() - mapping_middle)

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csr_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.float32)

            L = S @ L @ S.T
            R = S @ R @ S.T
            Mapping = Mapping @ S.T

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")

        if DEBUG:
            print("MATRIX_SIZE: ", MATRIX_SIZE)
            print("L: ", L)
            print("R: ", R)
            print("Mapping: ", Mapping)

    if DEBUG:
        _view_operations(operations)

    exec_time = time.time() - START_TIME

    if MODE == 'strong_simulation':
        # print(Mapping.getrow(-2).indices.tolist())
        sim_result, sim_time = simulation_all_path(L, R, Mapping.getrow(-2).indices.tolist(), NUM_QUBITS, TYPE, START_TIME)
    elif MODE == 'weak_simulation':
        sim_result, sim_time = weak_simulation(L, R, Mapping.toarray(), NUM_QUBITS, TYPE, START_TIME, num_shots=1024)

    # sim_result = 0
    return exec_time, sim_result, sim_time

In [130]:
def dd_sparse_cpu(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int, TYPE: str, START_TIME: float, MODE: str = 'strong_simulation', DEBUG = False):
    START_TIME = time.time()
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS, TYPE)
    # print("Init Mapping: ", Mapping)
    # print("L:", L)
    # print("R:", R)

    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse

    for op in operations:
        if time.time() - START_TIME > 60:
            raise TimeoutError("Timeout in Circuit Operations: 60s.")

        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)

        target_map = Mapping.getrow(op.target).indices.tolist()

        A = cp.ones(MATRIX_SIZE, dtype=cp.complex128)
        A[target_map] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=cp.complex128)
        B[target_map] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=cp.complex128)
        C[target_map] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=cp.complex128)
        D[target_map] = op.d

        A = sparse.diags(A, offsets=0, format='csr')
        B = sparse.diags(B, offsets=0, format='csr')
        C = sparse.diags(C, offsets=0, format='csr')
        D = sparse.diags(D, offsets=0, format='csr')


        L_U = L @ A + R @ B
        R_U = L @ C + R @ D
        Mapping_U = Mapping.copy()

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)

        elif isinstance(op, ControlledGate):
            MATRIX_SIZE_OLD = MATRIX_SIZE

            # print("L: ", L)
            # print("R: ", R)

            # L_U = L_U @ sparse.diags((1-Mapping[op.control].sum(axis=0).toarray().ravel()), offsets=0, format='csr')
            L_U = L_U.multiply((cp.asarray(Mapping[op.control].sum(axis=0)).ravel() == 0).astype(L_U.dtype)).tocsr()


            control_up = [control for control in op.control if control < op.target]
            control_down = [control for control in op.control if control > op.target]

            node_target = cp.asarray(Mapping[op.target].sum(axis=0)).ravel()
            # print("node_target: ", node_target)


            if len(control_up) != 0:
                node_up_all = cp.asarray(Mapping.sum(axis=0)).ravel() if op.target == NUM_QUBITS - 1 else cp.asarray(Mapping[[i for i in range(op.target+2)]].sum(axis=0)).ravel()
                node_up = cp.asarray(Mapping[[i for i in range(op.target)]].sum(axis=0)).ravel()
                node_control_up = cp.asarray(Mapping[control_up].sum(axis=0)).ravel()
                # print("node_up_all: ", node_up_all)
                # print("node_up: ", node_up)
                # print("node_control_up: ", node_control_up)
                node_up_not_zero = node_up_all != 0
                L_up = L[node_up_not_zero][:, node_up_not_zero]
                R_up = R[node_up_not_zero][:, node_up_not_zero]

                up_control = sparse.diags(node_control_up[:L_up.shape[0]], offsets=0, format='csr')
                up_target = sparse.diags(node_target[:L_up.shape[0]], offsets=0, format='csr')
                up = sparse.diags(node_up[:L_up.shape[0]], offsets=0, format='csr')

                # print("up_control: ", up_control)
                # print("up_target: ", up_target)
                # print("up: ", up)

                L_up_control = L_up @ up_control
                L_up_target = L_up @ up_target
                R_up_target = R_up @ up_target
                L_up = L_up @ up
                R_up = R_up @ up

                # print("L_U: ", L_U)

                # print("L_up_control: ", L_up_control)
                # print("L_up_target: ", L_up_target)
                # print("R_up_target: ", R_up_target)


                # M_U = sparse.diags((1-node_control_up), offsets=0, format='csr')
                # print("M_U: ", M_U)
                # L_U = L_U @ M_U

                print("L_U.shape: ", L_U.shape)
                print("L_up.shape: ", L_up.shape)
                print("L_up_target.shape: ", L_up_target.shape)
                print("L_up_control.shape: ", L_up_control.shape)
                print("R_U.shape: ", R_U.shape)
                print("R_up.shape: ", R_up.shape)
                print("R_up_target.shape: ", R_up_target.shape)
                L_U = sparse.bmat([[L_U[:L_up.shape[0], :L_up.shape[0]], L_U[:L_up.shape[0], L_up.shape[0]:], L_up_target],
                                [L_U[L_up.shape[0]:, :L_up.shape[0]], L_U[L_up.shape[0]:, L_up.shape[0]:], None],
                                [L_up_control, None, L_up]], format='csr')
                print("L_U PASS")
                R_U = sparse.bmat([[R_U[:R_up.shape[0], :], R_up_target],
                                [R_U[R_up.shape[0]:, :], None],
                                [None, R_up]], format='csr')
                print("R_U PASS")

                # print("L_U: ", L_U)
                # print("R_U: ", R_U)

                # print("L_U.shape: ", L_U.shape)

                # print("Mapping: ", Mapping)
                # print("Mapping[:, node_up_all]: ", Mapping[:, node_up_all])
                # print("Mapping[node_up, :]: ", Mapping[node_up, :])

                # print("cp.array(node_up_all, dtype=bool): ", cp.array(node_up_all, dtype=bool))
                # print("cp.where(cp.array(node_up_all, dtype=bool))[0]: ", cp.where(cp.array(node_up_all, dtype=bool))[0])
                # print("Mapping[:, cp.where(cp.array(node_up_all, dtype=bool))[0]]: ", Mapping[:, cp.where(cp.array(node_up_all, dtype=bool))[0]])
                # print("Mapping[:, cp.array(node_up_all, dtype=bool)]: ", Mapping[:, cp.array(node_up_all, dtype=bool)])

                mask = cp.asarray(node_up_all, dtype=bool)
                idx  = cp.flatnonzero(mask).astype(cp.int32)

                S_up = sparse.csr_matrix(
                    (cp.ones(idx.size, dtype=cp.complex128),
                    (idx, cp.arange(idx.size, dtype=cp.int32))),
                    shape=(Mapping.shape[1], int(idx.size))
                )

                Mapping_U = sparse.hstack([Mapping, Mapping@S_up], format='csr')
                # Mapping_U = sparse.hstack([Mapping, Mapping], format='csr') if op.target == NUM_QUBITS - 1 else sparse.hstack([Mapping[:, :op.target+2], Mapping[:, op.target+1:]], format='csr')
                # print("Mapping_U.shape: ", Mapping_U.shape)

            # print("Mapping: ", Mapping)
            # print("Mapping[:op.target+1]: ", Mapping[:, :op.target+1])
            # print("Mapping[op.target:]: ", Mapping[:, op.target:])

            if len(control_down) != 0:
                node_down = cp.asarray(Mapping[[i for i in range(op.target, control_down[-1])]].sum(axis=0)).ravel()
                node_down_all = cp.asarray(Mapping[[i for i in range(op.target, Mapping.shape[0])]].sum(axis=0)).ravel() if control_down[-1] == NUM_QUBITS - 1 else cp.asarray(Mapping[[i for i in range(op.target, control_down[-1]+2)]].sum(axis=0)).ravel()
                node_control_down = cp.asarray(Mapping[control_down].sum(axis=0)).ravel()

                node_down_all_not_zero = node_down_all != 0
                L_down = L[node_down_all_not_zero][:, node_down_all_not_zero]
                R_down = R[node_down_all_not_zero][:, node_down_all_not_zero]
                # print("L_down: ", L_down)
                # print("R_down: ", R_down)


                down = sparse.diags(node_down[-L_down.shape[0]:], offsets=0, format='csr')

                node_control_down_except_last = cp.asarray(Mapping[control_down[:-1]].sum(axis=0)).ravel()
                node_control_down_only_last = cp.asarray(Mapping[control_down[-1]].sum(axis=0)).ravel()
                # print("node_down: ", node_down)
                # print("node_down_all: ", node_down_all)
                # print("node_control_down: ", node_control_down)
                # print("node_control_down_except_last: ", node_control_down_except_last)
                # print("node_control_down_only_last: ", node_control_down_only_last)

                down_last_control = sparse.diags(node_control_down_only_last[-L_down.shape[0]:], offsets=0, format='csr')
                L_down_last_control = L_down @ down_last_control
                R_down_last_control = None

                L_down = L_down @ down
                R_down = R_down @ down

                down_other_control_neg = sparse.diags((1-node_control_down_except_last[-L_down.shape[0]:]), offsets=0, format='csr')
                L_down_all = L_down @ down_other_control_neg
                R_down_all = R_down.copy()

                mask = cp.asarray(node_down_all, dtype=bool)
                idx  = cp.flatnonzero(mask).astype(cp.int32)

                S_down = sparse.csr_matrix(
                    (cp.ones(idx.size, dtype=cp.complex128),
                    (idx, cp.arange(idx.size, dtype=cp.int32))),
                    shape=(Mapping.shape[1], int(idx.size))
                )

                Mapping_U = sparse.hstack([Mapping_U, Mapping@S_down], format='csr')

                if len(control_down) > 1:
                    L_down_control_go_left = L_down @ sparse.diags(node_control_down_except_last[-L_down.shape[0]:], offsets=0, format='csr')
                    # print("L_down_control_go_left: ", L_down_control_go_left)
                    # print("L_down_all.shape: ", L_down_all.shape)
                    # print("L_down.shape: ", L_down.shape)
                    # print("L_down_control_go_left.shape: ", L_down_control_go_left.shape)
                    L_down_all = sparse.bmat([[L_down_all, None],
                                        [L_down_control_go_left, L_down]], format='csr')
                    R_down_all = sparse.bmat([[R_down_all, None],
                                        [None, R_down]], format='csr')

                    # print("L_down_all: ", L_down_all)
                    # print("R_down_all: ", R_down_all)
                    L_down_last_control = sparse.hstack([L_down_last_control, L_down_last_control], format='csr')
                    R_down_last_control = sparse.hstack([R_down_last_control, R_down @ down_last_control], format='csr')
                    Mapping_U = sparse.hstack([Mapping_U, Mapping@S_down], format='csr')

                down_target = sparse.diags(node_target[-L_down.shape[0]:], offsets=0, format='csr')
                L_down_target = L_down @ down_target
                R_down_target = R_down @ down_target
                # print("L_down_target: ", L_down_target)
                # print("R_down_target: ", R_down_target)

                if len(control_up) == 0:
                    # print("L_U[:L_down.shape[0], :-L_down.shape[0]]: ", L_U[:L_down.shape[0], :-L_down.shape[0]])
                    # print("L_U[:L_down.shape[0], -L_down.shape[0]:]: ", L_U[:L_down.shape[0], -L_down.shape[0]:])
                    # print("L_down_last_control: ", L_down_last_control)
                    # print("L_down_target: ", L_down_target)
                    L_U = sparse.bmat([[L_U[:-L_down.shape[0], :-L_down.shape[0]], L_U[:-L_down.shape[0], -L_down.shape[0]:], None],
                                    [L_U[-L_down.shape[0]:, :-L_down.shape[0]], L_U[-L_down.shape[0]:, -L_down.shape[0]:], L_down_last_control],
                                    [None, L_down_target, L_down_all[:L_down.shape[0]]],
                                    [None, None, L_down_all[L_down.shape[0]:]]], format='csr')
                    R_U = sparse.bmat([[R_U[:-R_down.shape[0], :-R_down.shape[0]], R_U[:-R_down.shape[0], -R_down.shape[0]:], None],
                                    [R_U[-R_down.shape[0]:, :-R_down.shape[0]], R_U[-R_down.shape[0]:, -R_down.shape[0]:], R_down_last_control],
                                    [None, R_down_target, R_down_all[:R_down.shape[0]]],
                                    [None, None, R_down_all[R_down.shape[0]:]]], format='csr')
                else:
                    # print("L_U.shape[0]: ", L_U.shape[0])
                    # print("L_down.shape[0]: ", L_down.shape[0])
                    # print("MATRIX_SIZE: ", MATRIX_SIZE)
                    # print("L_down_last_control: ", L_down_last_control)
                    # print("L_down_all: ", L_down_all)
                    # print("L_U[:-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]]: ", L_U[:-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]])
                    # print("L_U[:-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE]: ", L_U[:-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE])
                    # print("L_U[:-L_down.shape[0], MATRIX_SIZE:L_U.shape[0]-L_down.shape[0]]: ", L_U[:-L_down.shape[0], MATRIX_SIZE:L_U.shape[0]-L_down.shape[0]])
                    # print("L_U[:-L_down.shape[0], L_U.shape[0]-L_down.shape[0]:]: ", L_U[:-L_down.shape[0], L_U.shape[0]-L_down.shape[0]:])
                    L_U = sparse.bmat([[L_U[:MATRIX_SIZE-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]], L_U[:MATRIX_SIZE-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[:MATRIX_SIZE-L_down.shape[0], MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[:MATRIX_SIZE-L_down.shape[0], 2*MATRIX_SIZE-L_down.shape[0]:], None],
                                    [L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, :MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, 2*MATRIX_SIZE-L_down.shape[0]:], L_down_last_control],
                                    [L_U[MATRIX_SIZE:, :MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE:, MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[MATRIX_SIZE:, MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE:, 2*MATRIX_SIZE-L_down.shape[0]:], None],
                                    [None, L_down_target, None, L_down_target[:, :L_U.shape[0]-2*MATRIX_SIZE+L_down.shape[0]], L_down_all[:L_down.shape[0]]],
                                    [None, None, None, None, L_down_all[L_down.shape[0]:]]], format='csr')
                    R_U = sparse.bmat([[R_U[:MATRIX_SIZE-R_down.shape[0], :MATRIX_SIZE-R_down.shape[0]], R_U[:MATRIX_SIZE-R_down.shape[0], MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[:MATRIX_SIZE-R_down.shape[0], MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[:MATRIX_SIZE-R_down.shape[0], 2*MATRIX_SIZE-R_down.shape[0]:], None],
                                    [R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, :MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, 2*MATRIX_SIZE-R_down.shape[0]:], R_down_last_control],
                                    [R_U[MATRIX_SIZE:, :MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE:, MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[MATRIX_SIZE:, MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE:, 2*MATRIX_SIZE-R_down.shape[0]:], None],
                                    [None, R_down_target, None, R_down_target[:, :R_U.shape[0]-2*MATRIX_SIZE+R_down.shape[0]], R_down_all[:R_down.shape[0]]],
                                    [None, None, None, None, R_down_all[R_down.shape[0]:]]], format='csr')

                    # L_U = sparse.bmat([[L_U[:L_down.shape[0], :-L_down.shape[0]], L_U[:L_down.shape[0], -L_down.shape[0]:MATRIX_SIZE], L_U[:L_down.shape[0], MATRIX_SIZE:MATRIX_SIZE+(-L_down.shape[0])], L_U[:L_down.shape[0], MATRIX_SIZE+(-L_down.shape[0]):], L_down_back],
                    #                 [L_U[L_down.shape[0]:, :-L_down.shape[0]], L_U[L_down.shape[0]:, -L_down.shape[0]:MATRIX_SIZE], L_U[L_down.shape[0]:, MATRIX_SIZE:MATRIX_SIZE+(-L_down.shape[0])], L_U[L_down.shape[0]:, MATRIX_SIZE+(-L_down.shape[0]):], None],
                    #                 [None, L_down_target, None, L_down_target, L_down_all[:L_down.shape[0]]],
                    #                 [None, None, None, None, L_down_all[L_down.shape[0]:]]], format='csr')

                    # R_U = sparse.bmat([[R_U[:R_down.shape[0], :-R_down.shape[0]], R_U[:R_down.shape[0], -R_down.shape[0]:MATRIX_SIZE], R_U[:R_down.shape[0], MATRIX_SIZE:MATRIX_SIZE+(-R_down.shape[0])], R_U[:R_down.shape[0], MATRIX_SIZE+(-R_down.shape[0]):], R_down_back],
                    #                 [R_U[R_down.shape[0]:, :-R_down.shape[0]], R_U[R_down.shape[0]:, -R_down.shape[0]:MATRIX_SIZE], R_U[R_down.shape[0]:, MATRIX_SIZE:MATRIX_SIZE+(-R_down.shape[0])], R_U[R_down.shape[0]:, MATRIX_SIZE+(-R_down.shape[0]):], None],
                    #                 [None, R_down_target, None, R_down_target, R_down_all[:R_down.shape[0]]],
                    #                 [None, None, None, None, R_down_all[R_down.shape[0]:]]], format='csr')

            if False:
                for control in control_down:
                    keep_U = cp.ones(MATRIX_SIZE, dtype=int)
                    keep = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                    map_control = Mapping.getrow(control).indices.tolist()
                    keep_U[map_control] = 0
                    keep[map_control] = 0
                    M_U = sparse.diags(keep_U, offsets=0, format='csr')
                    M = sparse.diags(keep, offsets=0, format='csr')

                    # print("M_U: ", M_U)
                    # print("M: ", M)
                    # print("L: ", L)
                    # print("L[1:, 0]: ", L[1:, 0].shape)
                    # print("L[1:, 1:]: ", L[1:, 1:].shape)
                    # print("L_U @ M_U: ", (L_U @ M_U).shape)

                    L_U_old = L_U @ M_U

                    L_U = sparse.bmat([[L_U_old[:, 0:1], L_U_old[:, 1:], None, None],
                                    [L[:, 0:1], None, None, L[:, :]]], format='csr')

                    R_old = R @ M
                    # print("R_old: ", R_old)
                    # print("R_old[1:, 0]: ", R_old[1:, 0:1])
                    # print("R_old[1:, 1:]: ", R_old[1:, 1:])

                    R_U = sparse.bmat([[R_U[:, 0:1], R_U[:, 1:], None, None],
                                    [R_old[:, 0:1], None, None, R_old[:, :]]], format='csr')



            if False:
                if control < op.target:
                    R_masked = R @ M
                    R_U_masked = R_U[:, :MATRIX_SIZE_OLD] @ M_U

                    L_U = sparse.bmat([[L, None],
                        [None, L_U]], format='csr')

                    # print("Mapping: ", Mapping[control])

                    # print("D: ", D)
                    # print("D_complement: ", D_complement)
                    # print("R: ", R)
                    # print("R_U: ", R_U)
                    # print("R_U[]: ", R_U[:, :MATRIX_SIZE_OLD])
                    # print("R_masked: ", R_masked)
                    # print("R_U_masked: ", R_U_masked)


                    # if TYPE == 'cupy':
                    #     print("R Masked Type: ", type(R_masked))
                    #     print("R U Masked Type: ", type(R_U_masked))
                    #     print("R U Type: ", type(R_U))
                    #     print("R Masked shape: ", R_masked.shape)
                    #     print("R U Masked shape: ", R_U_masked.shape)
                    #     print("R_U shape: ", R_U.shape)
                        # print("R_masked max row/col:", R_masked.nonzero())
                        # print("R_U_masked max row/col:", R_U_masked.nonzero())
                        # print("R_U max row/col:", R_U.nonzero())

                    R_U = sparse.bmat([[R_masked, None],
                        [R_U_masked, R_U]], format='csr')

                    # print("L_U: ", L_U)
                    # print("R_U: ", R_U)
                else:

                    L_U_masked = L_U @ D_complement
                    L_masked = L[:, :MATRIX_SIZE_OLD] @ D
                    L_U = sparse.bmat([[L_U_masked, None],
                        [L_masked, L]], format='csr')

                    R_U = sparse.bmat([[R_U, None],
                        [None, R]], format='csr')

                    MATRIX_SIZE += MATRIX_SIZE_OLD
                    Mapping_U = sparse.hstack([Mapping_U, Mapping], format='csr')

            L, R, Mapping, MATRIX_SIZE = L_U, R_U, Mapping_U, L_U.shape[0]
            # print("L: ", L)
            # print("R: ", R)
            # print("start traverse")
            if DEBUG:
                print("L: ", L)
                print("R: ", R)
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)

            traverse = sparse.csr_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)

            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            # Mapping_U = [
            #     sorted(
            #         [n + MATRIX_SIZE_OLD * i
            #         for i in range(1, int(MATRIX_SIZE / MATRIX_SIZE_OLD) + 1)
            #         for n in mapping
            #         if n + MATRIX_SIZE_OLD * i in traverse] +
            #         [n for n in mapping if n in traverse]
            #     )
            #     for mapping in Mapping
            # ]
            # print("Mapping_U: ", Mapping_U)

            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            # Mapping = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U after dict: ", Mapping_U)
            # Mapping = copy.deepcopy(Mapping_U)
            # print("end mapping, time spent:", time.time() - mapping_middle)

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csr_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.complex128)

            L = S @ L @ S.T
            R = S @ R @ S.T

            # print("S.shape: ", S.shape)
            # print("Mapping.shape: ", Mapping.shape)
            Mapping = Mapping @ S.T

            # print("End Mapping.shape: ", Mapping.shape)

            m = Mapping.shape[1]
            Mapping = Mapping.tocsc()[:, cp.arange(m, dtype=cp.int32)].tocsr()

            # print("End Mapping: ", Mapping)

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")

        if DEBUG:
            print("MATRIX_SIZE: ", MATRIX_SIZE)
            print("L: ", L)
            print("R: ", R)
            print("Mapping: ", Mapping)

    if DEBUG:
        _view_operations(operations)

    exec_time = time.time() - START_TIME

    if MODE == 'strong_simulation':
        # print(MATRIX_SIZE, Mapping.getrow(-2).indices.tolist())
        sim_result, sim_time = simulation_all_path(L, R, Mapping.getrow(-2).indices.tolist(), NUM_QUBITS, TYPE, START_TIME)
    elif MODE == 'weak_simulation':
        sim_result, sim_time = weak_simulation(L, R, Mapping.toarray(), NUM_QUBITS, TYPE, START_TIME, num_shots=1024)

    # sim_result = 0
    return exec_time, sim_result, sim_time


In [131]:
def dd_sparse_cuda(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int, TYPE: str, START_TIME: float, MODE: str = 'strong_simulation', DEBUG = False):
    START_TIME = time.time()
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS, TYPE)
    # print("Init Mapping: ", Mapping)
    # print("L:", L)
    # print("R:", R)

    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
    
    for op in operations:
        if time.time() - START_TIME > 60:
            raise TimeoutError("Timeout in Circuit Operations: 60s.")
        
        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        # L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)

        target_map = Mapping.getrow(op.target).indices.tolist()

        A = cp.ones(MATRIX_SIZE, dtype=cp.complex128)
        A[target_map] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=cp.complex128)
        B[target_map] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=cp.complex128)
        C[target_map] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=cp.complex128)
        D[target_map] = op.d

        A = sparse.diags(A, offsets=0, format='csr')
        B = sparse.diags(B, offsets=0, format='csr')
        C = sparse.diags(C, offsets=0, format='csr')
        D = sparse.diags(D, offsets=0, format='csr')
     

        L_U = L @ A + R @ B
        R_U = L @ C + R @ D
        Mapping_U = Mapping.copy()

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)
        
        elif isinstance(op, ControlledGate):
            MATRIX_SIZE_OLD = MATRIX_SIZE

            # print("L: ", L)
            # print("R: ", R)

            # L_U = L_U @ sparse.diags((1-Mapping[op.control].sum(axis=0).toarray().ravel()), offsets=0, format='csr')
            L_U = L_U.multiply((cp.asarray(Mapping[op.control].sum(axis=0)).ravel() == 0).astype(L_U.dtype)).tocsc()


            control_up = [control for control in op.control if control < op.target]
            control_down = [control for control in op.control if control > op.target]

            node_target = cp.asarray(Mapping[op.target].sum(axis=0)).ravel()
            # print("node_target: ", node_target)


            if len(control_up) != 0:
                node_up_all = cp.asarray(Mapping.sum(axis=0)).ravel() if op.target == NUM_QUBITS - 1 else cp.asarray(Mapping[[i for i in range(op.target+2)]].sum(axis=0)).ravel()
                node_up = cp.asarray(Mapping[[i for i in range(op.target)]].sum(axis=0)).ravel()
                node_control_up = cp.asarray(Mapping[control_up].sum(axis=0)).ravel()
                print("node_up_all: ", node_up_all)
                # print("node_up: ", node_up)
                # print("node_control_up: ", node_control_up)
                node_up_not_zero = node_up_all != 0
                L_up = L[node_up_not_zero][:, node_up_not_zero]
                R_up = R[node_up_not_zero][:, node_up_not_zero]

                up_control = sparse.diags(node_control_up[:L_up.shape[0]], offsets=0, format='csr')
                up_target = sparse.diags(node_target[:L_up.shape[0]], offsets=0, format='csr')
                up = sparse.diags(node_up[:L_up.shape[0]], offsets=0, format='csr')

                print("L: ", L)
                print("L_up: ", L_up)
                print("up_control: ", up_control)
                
                # print("up_target: ", up_target)
                # print("up: ", up)

                L_up_control = L_up @ up_control
                L_up_target = L_up @ up_target
                R_up_target = R_up @ up_target
                L_up = L_up @ up
                R_up = R_up @ up

                # print("L_U: ", L_U)

                # print("L_up_control: ", L_up_control)
                # print("L_up_target: ", L_up_target)
                # print("R_up_target: ", R_up_target)


                # M_U = sparse.diags((1-node_control_up), offsets=0, format='csr')
                # print("M_U: ", M_U)
                # L_U = L_U @ M_U

                print("L_U.shape: ", L_U.shape, " max: ", L_U.indices.max().item())
                print("L_up.shape: ", L_up.shape, " max: ", L_up.indices.max().item())
                print("L_up_target.shape: ", L_up_target.shape, " max: ", L_up_target.indices.max().item())
                print("L_up_control.shape: ", L_up_control.shape, " max: ", L_up_control.indices.max().item())
                print("L_up_control: ", L_up_control)
                print("R_U.shape: ", R_U.shape, " max: ", R_U.indices.max().item())
                print("R_up.shape: ", R_up.shape, " max: ", R_up.indices.max().item())
                print("R_up_target.shape: ", R_up_target.shape, " max: ", R_up_target.indices.max().item())
                
                L_U = sparse.bmat([[L_U[:L_up.shape[0], :L_up.shape[0]], L_U[:L_up.shape[0], L_up.shape[0]:], L_up_target],
                                [L_U[L_up.shape[0]:, :L_up.shape[0]], L_U[L_up.shape[0]:, L_up.shape[0]:], None],
                                [L_up_control, None, L_up]], format='csc')
                print("L_U PASS")
                R_U = sparse.bmat([[R_U[:R_up.shape[0], :], R_up_target],
                                [R_U[R_up.shape[0]:, :], None],
                                [None, R_up]], format='csc')
                print("R_U PASS")

                # print("L_U: ", L_U)
                # print("R_U: ", R_U)

                # print("L_U.shape: ", L_U.shape)

                # print("Mapping: ", Mapping)
                # print("Mapping[:, node_up_all]: ", Mapping[:, node_up_all])
                # print("Mapping[node_up, :]: ", Mapping[node_up, :])

                # print("cp.array(node_up_all, dtype=bool): ", cp.array(node_up_all, dtype=bool))
                # print("cp.where(cp.array(node_up_all, dtype=bool))[0]: ", cp.where(cp.array(node_up_all, dtype=bool))[0])
                # print("Mapping[:, cp.where(cp.array(node_up_all, dtype=bool))[0]]: ", Mapping[:, cp.where(cp.array(node_up_all, dtype=bool))[0]])
                # print("Mapping[:, cp.array(node_up_all, dtype=bool)]: ", Mapping[:, cp.array(node_up_all, dtype=bool)])

                mask = cp.asarray(node_up_all, dtype=bool)
                idx  = cp.flatnonzero(mask).astype(cp.int32)

                S_up = sparse.csr_matrix(
                    (cp.ones(idx.size, dtype=cp.complex128),
                    (idx, cp.arange(idx.size, dtype=cp.int32))),
                    shape=(Mapping.shape[1], int(idx.size))
                )

                Mapping_U = sparse.hstack([Mapping, Mapping@S_up], format='csr')
                # Mapping_U = sparse.hstack([Mapping, Mapping], format='csr') if op.target == NUM_QUBITS - 1 else sparse.hstack([Mapping[:, :op.target+2], Mapping[:, op.target+1:]], format='csr')
                # print("Mapping_U.shape: ", Mapping_U.shape)
            
            # print("Mapping: ", Mapping)
            # print("Mapping[:op.target+1]: ", Mapping[:, :op.target+1])
            # print("Mapping[op.target:]: ", Mapping[:, op.target:])

            if len(control_down) != 0:
                node_down = cp.asarray(Mapping[[i for i in range(op.target, control_down[-1])]].sum(axis=0)).ravel()
                node_down_all = cp.asarray(Mapping[[i for i in range(op.target, Mapping.shape[0])]].sum(axis=0)).ravel() if control_down[-1] == NUM_QUBITS - 1 else cp.asarray(Mapping[[i for i in range(op.target, control_down[-1]+2)]].sum(axis=0)).ravel()
                node_control_down = cp.asarray(Mapping[control_down].sum(axis=0)).ravel()

                node_down_all_not_zero = node_down_all != 0
                L_down = L[node_down_all_not_zero][:, node_down_all_not_zero]
                R_down = R[node_down_all_not_zero][:, node_down_all_not_zero]
                # print("L_down: ", L_down)
                # print("R_down: ", R_down)


                down = sparse.diags(node_down[-L_down.shape[0]:], offsets=0, format='csr')

                node_control_down_except_last = cp.asarray(Mapping[control_down[:-1]].sum(axis=0)).ravel()
                node_control_down_only_last = cp.asarray(Mapping[control_down[-1]].sum(axis=0)).ravel()
                # print("node_down: ", node_down)
                # print("node_down_all: ", node_down_all)
                # print("node_control_down: ", node_control_down)
                # print("node_control_down_except_last: ", node_control_down_except_last)
                # print("node_control_down_only_last: ", node_control_down_only_last)

                down_last_control = sparse.diags(node_control_down_only_last[-L_down.shape[0]:], offsets=0, format='csr')
                L_down_last_control = L_down @ down_last_control
                R_down_last_control = None

                L_down = L_down @ down
                R_down = R_down @ down

                down_other_control_neg = sparse.diags((1-node_control_down_except_last[-L_down.shape[0]:]), offsets=0, format='csr')
                L_down_all = L_down @ down_other_control_neg
                R_down_all = R_down.copy()

                mask = cp.asarray(node_down_all, dtype=bool)
                idx  = cp.flatnonzero(mask).astype(cp.int32)

                S_down = sparse.csr_matrix(
                    (cp.ones(idx.size, dtype=cp.complex128),
                    (idx, cp.arange(idx.size, dtype=cp.int32))),
                    shape=(Mapping.shape[1], int(idx.size))
                )

                Mapping_U = sparse.hstack([Mapping_U, Mapping@S_down], format='csr')

                if len(control_down) > 1:
                    L_down_control_go_left = L_down @ sparse.diags(node_control_down_except_last[-L_down.shape[0]:], offsets=0, format='csr')
                    # print("L_down_control_go_left: ", L_down_control_go_left)
                    # print("L_down_all.shape: ", L_down_all.shape)
                    # print("L_down.shape: ", L_down.shape)
                    # print("L_down_control_go_left.shape: ", L_down_control_go_left.shape)
                    L_down_all = sparse.bmat([[L_down_all, None],
                                        [L_down_control_go_left, L_down]], format='csr')
                    R_down_all = sparse.bmat([[R_down_all, None],
                                        [None, R_down]], format='csr')

                    # print("L_down_all: ", L_down_all)
                    # print("R_down_all: ", R_down_all)
                    L_down_last_control = sparse.hstack([L_down_last_control, L_down_last_control], format='csr')
                    R_down_last_control = sparse.hstack([R_down_last_control, R_down @ down_last_control], format='csr')
                    Mapping_U = sparse.hstack([Mapping_U, Mapping@S_down], format='csr')

                down_target = sparse.diags(node_target[-L_down.shape[0]:], offsets=0, format='csr')
                L_down_target = L_down @ down_target
                R_down_target = R_down @ down_target
                # print("L_down_target: ", L_down_target)
                # print("R_down_target: ", R_down_target)

                if len(control_up) == 0:
                    # print("L_U[:L_down.shape[0], :-L_down.shape[0]]: ", L_U[:L_down.shape[0], :-L_down.shape[0]])
                    # print("L_U[:L_down.shape[0], -L_down.shape[0]:]: ", L_U[:L_down.shape[0], -L_down.shape[0]:])
                    # print("L_down_last_control: ", L_down_last_control)
                    # print("L_down_target: ", L_down_target)
                    L_U = sparse.bmat([[L_U[:-L_down.shape[0], :-L_down.shape[0]], L_U[:-L_down.shape[0], -L_down.shape[0]:], None],
                                    [L_U[-L_down.shape[0]:, :-L_down.shape[0]], L_U[-L_down.shape[0]:, -L_down.shape[0]:], L_down_last_control],
                                    [None, L_down_target, L_down_all[:L_down.shape[0]]],
                                    [None, None, L_down_all[L_down.shape[0]:]]], format='csr')
                    R_U = sparse.bmat([[R_U[:-R_down.shape[0], :-R_down.shape[0]], R_U[:-R_down.shape[0], -R_down.shape[0]:], None],
                                    [R_U[-R_down.shape[0]:, :-R_down.shape[0]], R_U[-R_down.shape[0]:, -R_down.shape[0]:], R_down_last_control],
                                    [None, R_down_target, R_down_all[:R_down.shape[0]]],
                                    [None, None, R_down_all[R_down.shape[0]:]]], format='csr')
                else:
                    # print("L_U.shape[0]: ", L_U.shape[0])
                    # print("L_down.shape[0]: ", L_down.shape[0])
                    # print("MATRIX_SIZE: ", MATRIX_SIZE)
                    # print("L_down_last_control: ", L_down_last_control)
                    # print("L_down_all: ", L_down_all)
                    # print("L_U[:-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]]: ", L_U[:-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]])
                    # print("L_U[:-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE]: ", L_U[:-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE])
                    # print("L_U[:-L_down.shape[0], MATRIX_SIZE:L_U.shape[0]-L_down.shape[0]]: ", L_U[:-L_down.shape[0], MATRIX_SIZE:L_U.shape[0]-L_down.shape[0]])
                    # print("L_U[:-L_down.shape[0], L_U.shape[0]-L_down.shape[0]:]: ", L_U[:-L_down.shape[0], L_U.shape[0]-L_down.shape[0]:])
                    L_U = sparse.bmat([[L_U[:MATRIX_SIZE-L_down.shape[0], :MATRIX_SIZE-L_down.shape[0]], L_U[:MATRIX_SIZE-L_down.shape[0], MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[:MATRIX_SIZE-L_down.shape[0], MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[:MATRIX_SIZE-L_down.shape[0], 2*MATRIX_SIZE-L_down.shape[0]:], None],
                                    [L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, :MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE, 2*MATRIX_SIZE-L_down.shape[0]:], L_down_last_control],
                                    [L_U[MATRIX_SIZE:, :MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE:, MATRIX_SIZE-L_down.shape[0]:MATRIX_SIZE], L_U[MATRIX_SIZE:, MATRIX_SIZE:2*MATRIX_SIZE-L_down.shape[0]], L_U[MATRIX_SIZE:, 2*MATRIX_SIZE-L_down.shape[0]:], None],
                                    [None, L_down_target, None, L_down_target[:, :L_U.shape[0]-2*MATRIX_SIZE+L_down.shape[0]], L_down_all[:L_down.shape[0]]],
                                    [None, None, None, None, L_down_all[L_down.shape[0]:]]], format='csr')
                    R_U = sparse.bmat([[R_U[:MATRIX_SIZE-R_down.shape[0], :MATRIX_SIZE-R_down.shape[0]], R_U[:MATRIX_SIZE-R_down.shape[0], MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[:MATRIX_SIZE-R_down.shape[0], MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[:MATRIX_SIZE-R_down.shape[0], 2*MATRIX_SIZE-R_down.shape[0]:], None],
                                    [R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, :MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE, 2*MATRIX_SIZE-R_down.shape[0]:], R_down_last_control],
                                    [R_U[MATRIX_SIZE:, :MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE:, MATRIX_SIZE-R_down.shape[0]:MATRIX_SIZE], R_U[MATRIX_SIZE:, MATRIX_SIZE:2*MATRIX_SIZE-R_down.shape[0]], R_U[MATRIX_SIZE:, 2*MATRIX_SIZE-R_down.shape[0]:], None],
                                    [None, R_down_target, None, R_down_target[:, :R_U.shape[0]-2*MATRIX_SIZE+R_down.shape[0]], R_down_all[:R_down.shape[0]]],
                                    [None, None, None, None, R_down_all[R_down.shape[0]:]]], format='csr')

                    # L_U = sparse.bmat([[L_U[:L_down.shape[0], :-L_down.shape[0]], L_U[:L_down.shape[0], -L_down.shape[0]:MATRIX_SIZE], L_U[:L_down.shape[0], MATRIX_SIZE:MATRIX_SIZE+(-L_down.shape[0])], L_U[:L_down.shape[0], MATRIX_SIZE+(-L_down.shape[0]):], L_down_back],
                    #                 [L_U[L_down.shape[0]:, :-L_down.shape[0]], L_U[L_down.shape[0]:, -L_down.shape[0]:MATRIX_SIZE], L_U[L_down.shape[0]:, MATRIX_SIZE:MATRIX_SIZE+(-L_down.shape[0])], L_U[L_down.shape[0]:, MATRIX_SIZE+(-L_down.shape[0]):], None],
                    #                 [None, L_down_target, None, L_down_target, L_down_all[:L_down.shape[0]]],
                    #                 [None, None, None, None, L_down_all[L_down.shape[0]:]]], format='csr')

                    # R_U = sparse.bmat([[R_U[:R_down.shape[0], :-R_down.shape[0]], R_U[:R_down.shape[0], -R_down.shape[0]:MATRIX_SIZE], R_U[:R_down.shape[0], MATRIX_SIZE:MATRIX_SIZE+(-R_down.shape[0])], R_U[:R_down.shape[0], MATRIX_SIZE+(-R_down.shape[0]):], R_down_back],
                    #                 [R_U[R_down.shape[0]:, :-R_down.shape[0]], R_U[R_down.shape[0]:, -R_down.shape[0]:MATRIX_SIZE], R_U[R_down.shape[0]:, MATRIX_SIZE:MATRIX_SIZE+(-R_down.shape[0])], R_U[R_down.shape[0]:, MATRIX_SIZE+(-R_down.shape[0]):], None],
                    #                 [None, R_down_target, None, R_down_target, R_down_all[:R_down.shape[0]]],
                    #                 [None, None, None, None, R_down_all[R_down.shape[0]:]]], format='csr')

            if False:
                for control in control_down:
                    keep_U = cp.ones(MATRIX_SIZE, dtype=int)
                    keep = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                    map_control = Mapping.getrow(control).indices.tolist()
                    keep_U[map_control] = 0
                    keep[map_control] = 0
                    M_U = sparse.diags(keep_U, offsets=0, format='csr')
                    M = sparse.diags(keep, offsets=0, format='csr')

                    # print("M_U: ", M_U)
                    # print("M: ", M)
                    # print("L: ", L)
                    # print("L[1:, 0]: ", L[1:, 0].shape)
                    # print("L[1:, 1:]: ", L[1:, 1:].shape)
                    # print("L_U @ M_U: ", (L_U @ M_U).shape)

                    L_U_old = L_U @ M_U

                    L_U = sparse.bmat([[L_U_old[:, 0:1], L_U_old[:, 1:], None, None],
                                    [L[:, 0:1], None, None, L[:, :]]], format='csr')
                    
                    R_old = R @ M
                    # print("R_old: ", R_old)
                    # print("R_old[1:, 0]: ", R_old[1:, 0:1])
                    # print("R_old[1:, 1:]: ", R_old[1:, 1:])

                    R_U = sparse.bmat([[R_U[:, 0:1], R_U[:, 1:], None, None],
                                    [R_old[:, 0:1], None, None, R_old[:, :]]], format='csr')
                
               

            if False:
                if control < op.target:
                    R_masked = R @ M
                    R_U_masked = R_U[:, :MATRIX_SIZE_OLD] @ M_U

                    L_U = sparse.bmat([[L, None],
                        [None, L_U]], format='csr')
                
                    # print("Mapping: ", Mapping[control])
                    
                    # print("D: ", D)
                    # print("D_complement: ", D_complement)
                    # print("R: ", R)
                    # print("R_U: ", R_U)
                    # print("R_U[]: ", R_U[:, :MATRIX_SIZE_OLD])
                    # print("R_masked: ", R_masked)
                    # print("R_U_masked: ", R_U_masked)

                    
                    # if TYPE == 'cupy':
                    #     print("R Masked Type: ", type(R_masked))
                    #     print("R U Masked Type: ", type(R_U_masked))
                    #     print("R U Type: ", type(R_U))
                    #     print("R Masked shape: ", R_masked.shape)
                    #     print("R U Masked shape: ", R_U_masked.shape)
                    #     print("R_U shape: ", R_U.shape)
                        # print("R_masked max row/col:", R_masked.nonzero())
                        # print("R_U_masked max row/col:", R_U_masked.nonzero())
                        # print("R_U max row/col:", R_U.nonzero())

                    R_U = sparse.bmat([[R_masked, None],
                        [R_U_masked, R_U]], format='csr')
                    
                    # print("L_U: ", L_U)
                    # print("R_U: ", R_U)
                else:
                    
                    L_U_masked = L_U @ D_complement
                    L_masked = L[:, :MATRIX_SIZE_OLD] @ D
                    L_U = sparse.bmat([[L_U_masked, None],
                        [L_masked, L]], format='csr')

                    R_U = sparse.bmat([[R_U, None],
                        [None, R]], format='csr')
                    
                    MATRIX_SIZE += MATRIX_SIZE_OLD
                    Mapping_U = sparse.hstack([Mapping_U, Mapping], format='csr')

            L, R, Mapping, MATRIX_SIZE = L_U, R_U, Mapping_U, L_U.shape[0]
            # print("L: ", L)
            # print("R: ", R)
            # print("start traverse")
            if DEBUG:
                print("L: ", L)
                print("R: ", R)
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)

            traverse = sparse.csr_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)

            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            # Mapping_U = [
            #     sorted(
            #         [n + MATRIX_SIZE_OLD * i
            #         for i in range(1, int(MATRIX_SIZE / MATRIX_SIZE_OLD) + 1)
            #         for n in mapping
            #         if n + MATRIX_SIZE_OLD * i in traverse] +
            #         [n for n in mapping if n in traverse]
            #     )
            #     for mapping in Mapping
            # ]
            # print("Mapping_U: ", Mapping_U)

            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            # Mapping = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U after dict: ", Mapping_U)
            # Mapping = copy.deepcopy(Mapping_U)
            # print("end mapping, time spent:", time.time() - mapping_middle)  

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csr_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.complex128)

            L = S @ L @ S.T
            R = S @ R @ S.T

            # print("S.shape: ", S.shape)
            # print("Mapping.shape: ", Mapping.shape)
            Mapping = Mapping @ S.T

            # print("End Mapping.shape: ", Mapping.shape)

            m = Mapping.shape[1]
            Mapping = Mapping.tocsc()[:, cp.arange(m, dtype=cp.int32)].tocsr()

            # print("End Mapping: ", Mapping)

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")
        
        if DEBUG:
            print("MATRIX_SIZE: ", MATRIX_SIZE)
            print("L: ", L)
            print("R: ", R)
            print("Mapping: ", Mapping)
    
    if DEBUG:
        _view_operations(operations)

    exec_time = time.time() - START_TIME

    if MODE == 'strong_simulation':
        # print(MATRIX_SIZE, Mapping.getrow(-2).indices.tolist())
        sim_result, sim_time = simulation_all_path(L, R, Mapping.getrow(-2).indices.tolist(), NUM_QUBITS, TYPE, START_TIME)
    elif MODE == 'weak_simulation':
        sim_result, sim_time = weak_simulation(L, R, Mapping.toarray(), NUM_QUBITS, TYPE, START_TIME, num_shots=1024)
    
    # sim_result = 0
    return exec_time, sim_result, sim_time


In [132]:
def dd_sparse_numpy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int, MODE: str = 'strong_simulation', DEBUG: bool = False):

    def monitor_cpu_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []

        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)  # percent of a core
            mem = process.memory_info().rss / 1024**2  # in MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)
            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

    import numpy as cp
    import scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_cpu_usage, args=(stop_event, result_dict))

        np_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
            # if time.time() - np_start > 60:
            #     raise TimeoutError("Simulation Timeout: 60s.")
        exec_time, sim_result, sim_time = dd_sparse_cpu(operations, INIT_MATRIX_SIZE, NUM_QUBITS, TYPE='numpy', START_TIME=np_start, MODE=MODE)

        np_end = time.time()

        stop_event.set()
        monitor_thread.join()

        result_dict["exec_time"] = exec_time
        result_dict["sim_time"] = sim_time
        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = None

        if DEBUG: print(sim_result)
        # for comparison function)
        result_dict["sim_result"] = sim_result
        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))
    except Exception as e:
        np_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in numpy + scipy:", e)
        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = str(e)
        result_dict["exec_time"] = None
        result_dict["sim_time"] = None
        traceback.print_exc()

    # print("Time spent: ", np_end - np_start)
    return result_dict


In [133]:
def dd_sparse_cupy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int, MODE: str = 'strong_simulation', DEBUG: bool = False):
    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []

        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    import cupy as cp
    import cupyx.scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))

        cp_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
        #     if time.time() - cp_start > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")
        exec_time, sim_result, sim_time = dd_sparse_cuda(operations, INIT_MATRIX_SIZE, NUM_QUBITS, TYPE='cupy', START_TIME=cp_start, MODE=MODE, DEBUG=DEBUG)

        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))

        device_id = cp.cuda.runtime.getDevice()
        device_props = cp.cuda.runtime.getDeviceProperties(device_id)
        # print(f"Device {device_id} name:", device_props['name'].decode())
        # print(sim_result)
        # for comparison function
        result_dict["sim_result"] = sim_result

        result_dict["exec_time"] = exec_time
        result_dict["sim_time"] = sim_time
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = None

    except Exception as e:
        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in cupy computation:", e)
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = str(e)
        result_dict["exec_time"] = None
        result_dict["sim_time"] = None
        traceback.print_exc()


    # print("Time spent: ", cp_end - cp_start)
    return result_dict


In [134]:
# cudaq.set_target("nvidia")

# kernel = build_kernel_from_operations(NUM_QUBITS, operations)
# result = cudaq.sample(kernel)
# print(result)


In [135]:
def cudaq_sv(path: str, NUM_QUBITS, MODE: str = 'nvidia'):
    # NUM_QUBITS = min(NUM_QUBITS, 16)
    NUM_QUBITS = 0
    # output_path = Path(path + 'circuit_cudaq.py')
    # !npx q-convert -i {input_path} -s qasm -o {output_path} -d cudaq

    method_style_gates = {"x", "y", "z", "h", "rx", "ry", "rz", "t", "s", "swap"}

    def parse_qasm_to_cudaq_python(qasm_file: str, output_py_file: str):
        with open(qasm_file, 'r') as f:
            lines = f.readlines()

        cudaq_lines = [
            "import cudaq\n",
            f"cudaq.set_target('{MODE}')\n",
            "@cudaq.kernel\n",
            "def circuit():\n"
        ]
        indent = "    "

        for line in lines:
            line = line.strip()
            if not line or line.startswith("OPENQASM") or line.startswith("include") or line.startswith("//"):
                continue

            if match := re.match(r"qreg\s+(\w+)\[(\d+)\];", line):
                regname, size = match.groups()
                cudaq_lines.append(f"{indent}{regname} = cudaq.qvector({size})\n")
                continue

            if match := re.match(r"([a-zA-Z_][a-zA-Z0-9_]*)\s+(.*);", line):
                gate_raw, arglist = match.groups()
                args = [arg.strip() for arg in arglist.split(",") if arg.strip()]

                ctrl_count = len(gate_raw) - len(gate_raw.lstrip("c"))
                base_gate = gate_raw[ctrl_count:]

                if ctrl_count > 0:
                    controls = args[:ctrl_count]
                    targets = args[ctrl_count:]

                    if base_gate in method_style_gates and len(targets) == 1:
                        ctrl_list = f"[{', '.join(controls)}]"
                        cudaq_lines.append(
                            f"{indent}{base_gate}.ctrl({ctrl_list}, {targets[0]})\n"
                        )
                    else:
                        ctrl_tuple = ", ".join(controls)
                        lambda_call = f"{base_gate}({', '.join(targets)})"
                        cudaq_lines.append(
                            f"{indent}cudaq.control(({ctrl_tuple}), lambda: {lambda_call})\n"
                        )
                else:
                    if base_gate in method_style_gates:
                        cudaq_lines.append(f"{indent}{base_gate}({', '.join(args)})\n")
                    else:
                        cudaq_lines.append(f"{indent}# TODO: Unrecognized gate: {base_gate}({', '.join(args)})\n")

        cudaq_lines.append(f"\ncounts = cudaq.sample(circuit, shots_count={1024})\nprint(counts)")
        with open(output_py_file, 'w') as f:
            f.writelines(cudaq_lines)


    input_path = Path(path + 'circuit.qasm')
    output_path = Path(path + 'circuit_cudaq_new.py')
    parse_qasm_to_cudaq_python(input_path, output_path)

    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []

        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    import subprocess

    result_dict = {}
    stop_event = threading.Event()
    monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))
    timeout_secs = 60

    cudaq_start = time.time()
    monitor_thread.start()

    try:
        result = subprocess.run(
            ['python', output_path],
            capture_output=True,
            text=True,
            check=True,
            timeout=timeout_secs
        )
        cudaq_end = time.time()
        result_dict['sim_result'] = result.stdout
        stop_event.set()
        monitor_thread.join()
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict['error'] = None

    except subprocess.TimeoutExpired as e:
        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict['sim_result'] = str(e.stderr)
        result_dict['error'] = f"Timeout after {timeout_secs}s"
        print(f"Timed out after {timeout_secs}s")


    except subprocess.CalledProcessError as e:
        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict['sim_result'] = str(e.stderr)
        result_dict['error'] = str(e.stderr)
        print("Error output:\n", e.stderr)
        traceback.print_exc()


    return result_dict

In [136]:
dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')

node_up_all:  [1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j]
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 9 stored elements and shape (9, 9)>
  Coords	Values
  (1, 0)	(1+0j)
  (3, 1)	(0.7071067811865475+0j)
  (4, 1)	(0.7071067811865475+0j)
  (4, 2)	(1.414213562373095+0j)
  (5, 3)	(0.7071067811865475+0j)
  (6, 3)	(0.7071067811865475+0j)
  (6, 4)	(1.414213562373095+0j)
  (7, 5)	(1+0j)
  (8, 6)	(1+0j)
L_up:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 9 stored elements and shape (9, 9)>
  Coords	Values
  (1, 0)	(1+0j)
  (3, 1)	(0.7071067811865475+0j)
  (4, 1)	(0.7071067811865475+0j)
  (4, 2)	(1.414213562373095+0j)
  (5, 3)	(0.7071067811865475+0j)
  (6, 3)	(0.7071067811865475+0j)
  (6, 4)	(1.414213562373095+0j)
  (7, 5)	(1+0j)
  (8, 6)	(1+0j)
up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 4 stored elements and shape (9, 9)>
  Coords	Values
  (1, 1)	(1+0j)
  (2, 2)	(1+0j)
  (3, 3)	(1+0j)
  (4, 4)	(

/mnt/c/Users/Yu-Hsuan Wu/DD-Matrix/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


{'avg_cpu': 68.7,
 'peak_mem': 827.52734375,
 'avg_gpu': 26.333333333333332,
 'peak_gpu_mem': 424.86328125,
 'sim_result': {(0, 1, 1, 1): array(0.5+0.j),
  (0, 1, 0, 0): array(0.5+0.j),
  (0, 0, 1, 0): array(0.5+0.j),
  (0, 0, 0, 0): array(0.5+0.j)},
 'exec_time': 0.696983814239502,
 'sim_time': 0.5167653560638428,
 'time_spent': 1.245607614517212,
 'error': None}

In [137]:
import pandas as pd
import openpyxl

manual =  False

benchmarks = {
            #   'BV': 1,
            # #   'BVSym': 1,
            # #   'GHZall': 2,
            #   'GHZzero': 2,
            #   'Grover': 2,
            # #   'GroverSym': 2,
            # #   'H2': 1,
            # #   'HXH': 1,
            #   'MCToffoli': 3,
            # #   'MOBV_reorder': 1,
            # #   'MOGrover': 3,
            # #   'MOGroverSym': 3,
              'OEGrover': 2
              }
MQT_benchmarks = {
              'DJ',
              'QAOA',
              'QFT',
              'QNN',
              'renamed_shor',
              'VQE'
              }
benchmark_data = {}

for benchmark in benchmarks:

    if manual:
        break

    methods = ['NumPy + PySparse + Weak Sim', 'NumPy + PySparse + Strong Sim', 'CuPy + cuSparse + Weak Sim', 'CuPy + cuSparse + Strong Sim', 'CudaQ (StateVec)', 'CudaQ (TensorNet)']
    subfields = ['Time (s)', 'Execution Time (s)', 'Simulation Time (s)', 'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error']

    columns = [('Qubits', '')]
    rows = []

    for method in methods:
        for sub in subfields:
            if (method == 'NumPy + PySparse + Weak Sim' or method == 'NumPy + PySparse + Strong Sim') and sub == 'GPU Memory Usage (MB)':
                continue
            if (method == 'CudaQ (StateVec)' or method == 'CudaQ (TensorNet)') and (sub == 'Execution Time (s)' or sub == 'Simulation Time (s)'):
                continue
            columns.append((method, sub))

    multi_columns = pd.MultiIndex.from_tuples(columns)

    flag = (True, True, True, True, True, True)

    while (flag[0] or flag[1] or flag[2] or flag[3]) and (flag[4] or flag[5]) and benchmarks[benchmark] <= 1000:

        if benchmark == 'Grover' and benchmarks[benchmark] == 8:
            break

        # if benchmark == 'GroverSym' and benchmarks[benchmark] == 22:
        #     break

        # if benchmark == 'MOGrover' and benchmarks[benchmark] == 12:
        #     break

        # if benchmark == 'MOGroverSym' and benchmarks[benchmark] == 12:
        #     break


        i = str(benchmarks[benchmark])
        if int(i) < 10:
            i = '0' + i
        if benchmark in ['GHZall', 'GHZzero', 'H2'] and int(i) < 100:
            i = '0' + i

        path = '../ext/AutoQ/benchmarks/all/' + benchmark + f'/{i}/'

        input_path = Path(path + 'circuit.qasm')
        NUM_QUBITS, operations = parse_qasm_to_custom(input_path)

        INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1
        # if NUM_QUBITS > 50:
        #     break
        print(f"Qubit count: {NUM_QUBITS}")
        # print("Operations:", operations)

        if flag[0]:
            np_weak_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
        if flag[1]:
            cp_weak_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
        if flag[2]:
            cudaq_sv_result = cudaq_sv(path, NUM_QUBITS, MODE='nvidia')
        if flag[3]:
            np_strong_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
        if flag[4]:
            cp_strong_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
        if flag[5]:
            cudaq_tn_result = cudaq_sv(path, NUM_QUBITS, MODE='tensornet')

        rows.append([NUM_QUBITS,
                     np_weak_result['time_spent'] if flag[0] else '--',
                     np_weak_result['exec_time'] if flag[0] else '--',
                     np_weak_result['sim_time'] if flag[0] else '--',
                     np_weak_result['peak_mem'] if flag[0] else '--',
                     np_weak_result['error'] if flag[0] else '--',
                     np_strong_result['time_spent'] if flag[3] else '--',
                     np_strong_result['exec_time'] if flag[3] else '--',
                     np_strong_result['sim_time'] if flag[3] else '--',
                     np_strong_result['peak_mem'] if flag[3] else '--',
                     np_strong_result['error'] if flag[3] else '--',
                     cp_weak_result['time_spent'] if flag[1] else '--',
                     cp_weak_result['exec_time'] if flag[1] else '--',
                     cp_weak_result['sim_time'] if flag[1] else '--',
                     cp_weak_result['peak_mem'] if flag[1] else '--',
                     cp_weak_result['peak_gpu_mem'] if flag[1] else '--',
                     cp_weak_result['error'] if flag[1] else '--',
                     cp_strong_result['time_spent'] if flag[4] else '--',
                     cp_strong_result['exec_time'] if flag[4] else '--',
                     cp_strong_result['sim_time'] if flag[4] else '--',
                     cp_strong_result['peak_mem'] if flag[4] else '--',
                     cp_strong_result['peak_gpu_mem'] if flag[4] else '--',
                     cp_strong_result['error'] if flag[4] else '--',
                     cudaq_sv_result['time_spent'] if flag[2] else '--',
                     cudaq_sv_result['peak_mem'] if flag[2] else '--',
                     cudaq_sv_result['peak_gpu_mem'] if flag[2] else '--',
                     cudaq_sv_result['error'] if flag[2] else '--',
                     cudaq_tn_result['time_spent'] if flag[5] else '--',
                     cudaq_tn_result['peak_mem'] if flag[5] else '--',
                     cudaq_tn_result['peak_gpu_mem'] if flag[5] else '--',
                     cudaq_tn_result['error'] if flag[5] else '--'
                     ])


        if not np_weak_result['error'] is None:
            flag = (False, flag[1], flag[2], flag[3], flag[4], flag[5])

        if not cp_weak_result['error'] is None:
            flag = (flag[0], False, flag[2], flag[3], flag[4], flag[5])

        if not cudaq_sv_result['error'] is None:
            flag = (flag[0], flag[1], False, flag[3], flag[4], flag[5])

        if not np_strong_result['error'] is None:
            flag = (flag[0], flag[1], flag[2], False, flag[4], flag[5])

        if not cp_strong_result['error'] is None:
            flag = (flag[0], flag[1], flag[2], flag[3], False, flag[5])

        if not cudaq_tn_result['error'] is None:
            flag = (flag[0], flag[1], flag[2], flag[3], flag[4], False)
        benchmarks[benchmark] += 1

    benchmark_data[benchmark] = rows


Qubit count: 3
L_U.shape:  (7, 7)
L_up.shape:  (5, 5)
L_up_target.shape:  (5, 5)
L_up_control.shape:  (5, 5)
R_U.shape:  (7, 7)
R_up.shape:  (5, 5)
R_up_target.shape:  (5, 5)
L_U PASS
R_U PASS
L_U.shape:  (7, 7)
L_up.shape:  (5, 5)
L_up_target.shape:  (5, 5)
L_up_control.shape:  (5, 5)
R_U.shape:  (7, 7)
R_up.shape:  (5, 5)
R_up_target.shape:  (5, 5)
L_U PASS
R_U PASS


/mnt/c/Users/Yu-Hsuan Wu/DD-Matrix/.venv/lib/python3.12/site-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


node_up_all:  [1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 0.+0.j 0.+0.j]
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (7, 7)>
  Coords	Values
  (2, 0)	(1+0j)
  (3, 1)	(1+0j)
  (4, 2)	(1+0j)
  (5, 3)	(1+0j)
  (6, 4)	(1+0j)
L_up:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (5, 5)>
  Coords	Values
  (2, 0)	(1+0j)
  (3, 1)	(1+0j)
  (4, 2)	(1+0j)
up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 1 stored elements and shape (5, 5)>
  Coords	Values
  (0, 0)	(1+0j)
L_U.shape:  (7, 7)  max:  6
L_up.shape:  (5, 5)  max:  0
L_up_target.shape:  (5, 5)  max:  2
L_up_control.shape:  (5, 5)  max:  0
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 1 stored elements and shape (5, 5)>
  Coords	Values
  (2, 0)	(1+0j)
R_U.shape:  (7, 7)  max:  4
R_up.shape:  (5, 5)  max:  0
R_up_target.shape:  (5, 5)  max:  2
L_U PASS
R_U PASS


/mnt/c/Users/Yu-Hsuan Wu/DD-Matrix/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


node_up_all:  [1.+0.j 1.+0.j 1.+0.j 1.+0.j 0.+0.j 0.+0.j 1.+0.j]
L:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 7 stored elements and shape (7, 7)>
  Coords	Values
  (1, 0)	(0.7071067811865475+0j)
  (2, 1)	(0.7071067811865475+0j)
  (3, 1)	(0.7071067811865475+0j)
  (3, 6)	0j
  (4, 2)	(1+0j)
  (5, 3)	(1+0j)
  (6, 0)	(-0.7071067811865475+0j)
L_up:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (5, 5)>
  Coords	Values
  (1, 0)	(0.7071067811865475+0j)
  (2, 1)	(0.7071067811865475+0j)
  (3, 1)	(0.7071067811865475+0j)
  (3, 4)	0j
  (4, 0)	(-0.7071067811865475+0j)
up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 1 stored elements and shape (5, 5)>
  Coords	Values
  (0, 0)	(1+0j)
L_U.shape:  (7, 7)  max:  6
L_up.shape:  (5, 5)  max:  0
L_up_target.shape:  (5, 5)  max:  1
L_up_control.shape:  (5, 5)  max:  0
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 2 stored ele

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pandas as pd
from typing import Dict, Tuple, Union, Iterable, Optional, Sequence

DEFAULT_METHODS = [
    'NumPy + PySparse + Weak Sim',
    'NumPy + PySparse + Strong Sim',
    'CuPy + cuSparse + Weak Sim',
    'CuPy + cuSparse + Strong Sim',
    'CudaQ (StateVec)',
    'CudaQ (TensorNet)'
]
DEFAULT_SUBFIELDS = [
    'Time (s)', 'Execution Time (s)', 'Simulation Time (s)',
    'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error'
]
def build_multi_columns(methods=None, subfields=None) -> pd.MultiIndex:
    methods = methods or DEFAULT_METHODS
    subfields = subfields or DEFAULT_SUBFIELDS
    cols = [('Qubits', '')]
    for method in methods:
        for sub in subfields:
            if (method in ['NumPy + PySparse + Weak Sim', 'NumPy + PySparse + Strong Sim']) and sub == 'GPU Memory Usage (MB)':
                continue
            if (method in ['CudaQ (StateVec)', 'CudaQ (TensorNet)']) and (sub in ['Execution Time (s)', 'Simulation Time (s)']):
                continue
            cols.append((method, sub))
    return pd.MultiIndex.from_tuples(cols)

def execute_qasm_case(
    qasm_path: Union[str, Path],
    flag: Tuple[bool, bool, bool, bool, bool, bool],
    *,
    cudaq_dir: Optional[Union[str, Path]] = None,
    methods=DEFAULT_METHODS,
    subfields=DEFAULT_SUBFIELDS,
    verbose: bool = False,
    parser=None,                              # optional single parser
    parser_map: Optional[Dict[str, callable]] = None  # optional {".ext": parse_fn}
) -> Tuple[list, Tuple[bool, bool, bool, bool, bool, bool]]:
    """
    Run one circuit file through your pipeline and return (row, updated_flags).

    If parser_map is provided and contains qasm_path.suffix, that function is used.
    Else if parser is provided, that is used.
    Else falls back to parse_qasm_to_custom.
    """
    methods = methods or DEFAULT_METHODS
    subfields = subfields or DEFAULT_SUBFIELDS
    qasm_path = Path(qasm_path)
    if cudaq_dir is None:
        cudaq_dir = qasm_path.parent
    cudaq_dir = Path(cudaq_dir)
    cudaq_dir_str = str(cudaq_dir if str(cudaq_dir).endswith('/') else str(cudaq_dir) + '/')

    # --- choose parser ---
    parse_fn = None
    if parser_map and qasm_path.suffix in parser_map:
        parse_fn = parser_map[qasm_path.suffix]
    elif parser is not None:
        parse_fn = parser
    else:
        parse_fn = parse_qasm_to_custom  # default

    # Parse
    try:
        NUM_QUBITS, operations = parse_fn(qasm_path)
    except Exception as e:
        row = [
            '--',
            '--', '--', '--', '--', f'parse-error: {e}',
            '--', '--', '--', '--', f'parse-error: {e}',
            '--', '--', '--', '--', '--', f'parse-error: {e}',
            '--', '--', '--', '--', '--', f'parse-error: {e}',
            '--', '--', '--', f'parse-error: {e}',
            '--', '--', '--', f'parse-error: {e}',
        ]
        return row, flag  # leave flags unchanged on parse error

    if verbose:
        print(f"[run] {qasm_path}  | qubits={NUM_QUBITS}")

    INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

    if flag[0]:
        np_weak_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
    if flag[1]:
        cp_weak_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
    if flag[2]:
        cudaq_sv_result = cudaq_sv(cudaq_dir_str, NUM_QUBITS, MODE='nvidia')
    if flag[3]:
        np_strong_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
    if flag[4]:
        cp_strong_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
    if flag[5]:
        cudaq_tn_result = cudaq_sv(cudaq_dir_str, NUM_QUBITS, MODE='tensornet')

    row = [
        NUM_QUBITS,
        np_weak_result['time_spent'] if flag[0] else '--',
        np_weak_result['exec_time']  if flag[0] else '--',
        np_weak_result['sim_time']   if flag[0] else '--',
        np_weak_result['peak_mem']   if flag[0] else '--',
        np_weak_result['error']      if flag[0] else '--',
        np_strong_result['time_spent'] if flag[3] else '--',
        np_strong_result['exec_time']  if flag[3] else '--',
        np_strong_result['sim_time']   if flag[3] else '--',
        np_strong_result['peak_mem']   if flag[3] else '--',
        np_strong_result['error']      if flag[3] else '--',
        cp_weak_result['time_spent']   if flag[1] else '--',
        cp_weak_result['exec_time']    if flag[1] else '--',
        cp_weak_result['sim_time']     if flag[1] else '--',
        cp_weak_result['peak_mem']     if flag[1] else '--',
        cp_weak_result['peak_gpu_mem'] if flag[1] else '--',
        cp_weak_result['error']        if flag[1] else '--',
        cp_strong_result['time_spent']   if flag[4] else '--',
        cp_strong_result['exec_time']    if flag[4] else '--',
        cp_strong_result['sim_time']     if flag[4] else '--',
        cp_strong_result['peak_mem']     if flag[4] else '--',
        cp_strong_result['peak_gpu_mem'] if flag[4] else '--',
        cp_strong_result['error']        if flag[4] else '--',
        cudaq_sv_result['time_spent']    if flag[2] else '--',
        cudaq_sv_result['peak_mem']      if flag[2] else '--',
        cudaq_sv_result['peak_gpu_mem']  if flag[2] else '--',
        cudaq_sv_result['error']         if flag[2] else '--',
        cudaq_tn_result['time_spent']    if flag[5] else '--',
        cudaq_tn_result['peak_mem']      if flag[5] else '--',
        cudaq_tn_result['peak_gpu_mem']  if flag[5] else '--',
        cudaq_tn_result['error']         if flag[5] else '--',
    ]

    # Update flags based on errors
    new_flag = (
        flag[0] and ((not flag[0]) or (np_weak_result['error']   is None)),
        flag[1] and ((not flag[1]) or (cp_weak_result['error']   is None)),
        flag[2] and ((not flag[2]) or (cudaq_sv_result['error']  is None)),
        flag[3] and ((not flag[3]) or (np_strong_result['error'] is None)),
        flag[4] and ((not flag[4]) or (cp_strong_result['error'] is None)),
        flag[5] and ((not flag[5]) or (cudaq_tn_result['error']  is None)),
    )
    return row, new_flag

def format_case_index(benchmark_name: str, k: int) -> str:
    i = f"{k:02d}"
    if benchmark_name in ['GHZall', 'GHZzero', 'H2'] and k < 100:
        i = f"{k:03d}"
    return i

def should_stop_early(benchmark_name: str, k: int) -> bool:
    if benchmark_name == 'Grover' and k == 8:
        return True
    return False

def run_autoq_benchmarks(
    benchmarks: dict,
    *,
    manual: bool = False,
    base_path: str = "../ext/AutoQ/benchmarks/all",
    methods=DEFAULT_METHODS,
    subfields=DEFAULT_SUBFIELDS,
    max_k: int = 1000,
    verbose: bool = True
) -> Tuple[Dict[str, list], pd.MultiIndex]:
    """Wraps your original outer loop using the shared executor."""
    multi_columns = build_multi_columns(methods, subfields)
    benchmark_data: Dict[str, list] = {}

    for name in benchmarks:
        if manual:
            break
        k = benchmarks[name]
        rows = []
        flag = (True, True, True, True, True, True)  # (npW, cpW, sv, npS, cpS, tn)

        while (flag[0] or flag[1] or flag[2] or flag[3]) and (flag[4] or flag[5]) and k <= max_k:
            if should_stop_early(name, k):
                break
            i = format_case_index(name, k)
            case_dir = f"{base_path}/{name}/{i}/"
            qasm_file = Path(case_dir) / "circuit.qasm"

            row, flag = execute_qasm_case(
                qasm_file,
                flag,
                cudaq_dir=case_dir,
                methods=methods,
                subfields=subfields,
                verbose=verbose,
            )
            rows.append(row)
            k += 1

        benchmark_data[name] = rows

    return benchmark_data, multi_columns

def run_benchmarks_from_tree(
    tree: Dict[str, Union[int, dict]],
    root: Union[str, Path],
    *,
    key_prefix: str = "VQB",
    file_globs: Union[str, Sequence[str]] = ("*.qasm", "*.real"),
    methods=DEFAULT_METHODS,
    subfields=DEFAULT_SUBFIELDS,
    verbose: bool = True,
    dry_run: bool = False,
    parser=None,
    parser_map: Optional[Dict[str, callable]] = None,
) -> Tuple[Dict[str, list], pd.MultiIndex]:

    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Root not found or not a directory: {root.resolve()}")

    def _iter_leaf_dirs(t: dict, base: Path, parts: Tuple[str, ...] = ()) -> Iterable[Path]:
        for name, sub in t.items():
            here = base.joinpath(*parts, name)
            if sub == 0:
                yield here
            elif isinstance(sub, dict):
                yield from _iter_leaf_dirs(sub, base, parts + (name,))

    multi_columns = build_multi_columns(methods, subfields)
    out: Dict[str, list] = {}
    total_files = 0
    leaf_dirs = list(_iter_leaf_dirs(tree, root))

    # normalize patterns
    patterns = (file_globs,) if isinstance(file_globs, str) else tuple(file_globs)

    for d in leaf_dirs:
        if not d.is_dir():
            continue

        # collect and dedupe matches across all patterns
        matches = []
        for pat in patterns:
            matches.extend(d.glob(pat))
        files = sorted({p.resolve() for p in matches}, key=lambda p: p.name.lower())
        if not files:
            continue

        if verbose:
            rel = d.relative_to(root).as_posix()
            print(f"[dir] {rel}  ({len(files)} file(s))")

        rows = []
        flag = (True, True, True, True, True, True)
        for fpath in files:
            total_files += 1
            if dry_run:
                rows.append(['--'] + ['--'] * (len(multi_columns) - 1))
                continue

            row, flag = execute_qasm_case(
                fpath,
                flag,
                cudaq_dir=d,
                methods=methods,
                subfields=subfields,
                verbose=verbose,
                parser=parser,
                parser_map=parser_map,
            )
            rows.append(row)

        key = f"{key_prefix}:{d.relative_to(root).as_posix()}"
        out[key] = rows

    if verbose:
        print(f"[done] VeriQBench processed {total_files} file(s) in {len(leaf_dirs)} leaf dir(s).")
    return out, multi_columns


In [ ]:
benchmarks = {
    'MCToffoli': 3,
    'OEGrover': 2,
}

autoq_data, autoq_cols = run_autoq_benchmarks(
    benchmarks,
    base_path="../ext/AutoQ/benchmarks/all",
    verbose=True,
)

pd.DataFrame(autoq_data['OEGrover'], columns=autoq_cols)

In [ ]:
from pathlib import Path

veriq = "../ext/VeriQBench"
mqt = "../ext/DD_benchmark/MQTbench"
revlib = "../ext/DD_benchmark/Revlib_renamed"

root = Path(mqt)

if not root.is_dir():
    raise FileNotFoundError(f"Root not found or not a directory: {root.resolve()}")

def build_tree_or_zero(dir_path: Path):
    """
    Return a nested dict of subfolders for dir_path.
    If a folder has no (non-symlink) subfolders, return 0 instead of {}.
    """
    children = {}
    for p in sorted(dir_path.iterdir(), key=lambda x: x.name.lower()):
        try:
            if p.is_symlink() or not p.is_dir():
                continue
        except OSError:
            continue
        children[p.name] = build_tree_or_zero(p)
    return 0 if not children else children

gen_tree = {
    d.name: build_tree_or_zero(d)
    for d in sorted(root.iterdir(), key=lambda x: x.name.lower())
    if d.is_dir() and not d.is_symlink()
}

gen_tree

In [ ]:
mqtbench_tree = {
  'DJ': 0,
  'QAOA': 0,
 	'QFT': 0,
 	'QNN': 0,
 	'renamed_shor': 0,
 	'Shor': 0,
 	'VQE': 0
}

mqt_data, mqt_cols = run_benchmarks_from_tree(
    mqtbench_tree,
    root="ext/DD_benchmark/MQTbench",
    key_prefix="MQT",
    file_glob="*.qasm",
    verbose=True,
)

# Example DF for a leaf:
leaf_key = "MQT:/DJ"
pd.DataFrame(mqt_data[leaf_key], columns=mqt_cols)


In [ ]:
revlib_tree = {
  '4GT_renamed': 0,
	'4MOD_renamed': 0,
	'ADD_renamed': 0,
	'ALU_renamed': 0,
	'APEX_renamed': 0,
	'CM_renamed': 0,
	'CYCLE_renamed': 0,
	'DECODE_renamed': 0,
	'HAM_renamed': 0,
	'HWB_renamed': 0,
	'MOD_renamed': 0,
	'ONE_renamed': 0,
	'OTHER_renamed': 0,
	'PLUS_renamed': 0,
	'RD_renamed': 0,
	'SYM_renamed': 0,
	'URF_renamed': 0
}

revlib_data, revlib_cols = run_benchmarks_from_tree(
    revlib_tree,
    root="ext/DD_benchmark/Revlib_renamed",
    key_prefix="RevLib",
    file_glob="*.real",
    verbose=True,
)

# Example DF for a leaf:
leaf_key = "RevLib:/4GT_renamed"
pd.DataFrame(revlib_data[leaf_key], columns=revlib_cols)

In [ ]:
veriqbench_tree = {
  'combinational': {
    'adder': 0,
    'bv': 0,
    'grover': 0,
    'pe': 0,
    'qft': 0,
    'qubit_mapping': {
			'Aspen-4': 0,
			'Ourense': 0,
			'Rochester': 0,
			'Sycamore': 0,
			'Tokyo': 0
    },
  	'qv': 0,
  	'rand_cliff': 0,
  	'rev_circuit': 0,
  	'supre_circuit': 0
  },
  'dynamic': {
    'pe': 0,
    'qft': 0
  },
  'hardware_experiments': 0,
  'scripts': {
    'combinational': 0,
    'dynamic': 0,
    'sequential': 0,
    'variational': 0
  },
  'sequential': 0,
  'variational': {
    'EfficientSU2': 0,
    'examples': 0,
    'HFVQE': 0,
    'QAOA': 0,
    'QCNN': 0
  }
}


veriq_data, veriq_cols = run_benchmarks_from_tree(
    veriqbench_tree,
    root="ext/VeriQBench",
    key_prefix="VQB",
    file_glob="*.qasm",
    verbose=True,
)

# Example DF for a leaf:
leaf_key = "VQB:combinational/adder"
pd.DataFrame(veriq_data[leaf_key], columns=veriq_cols)


In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

excel_output_path = '../result/complex.xlsx'

excel_file = Path(excel_output_path)

def write_sheets(writer, data_dict):
    for benchmark, rows in data_dict.items():
        df = pd.DataFrame(rows, columns=multi_columns)
        df.to_excel(writer, sheet_name=benchmark, index=True)

if not excel_file.exists():
    # Create a new Excel file if it does not exist
    with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
        write_sheets(writer, benchmark_data)
else:
    try:
        # Load the workbook
        book = load_workbook(excel_output_path)
        for benchmark in benchmark_data.keys():
            if benchmark in book.sheetnames:
                # Delete the sheet if it exists
                std = book[benchmark]
                book.remove(std)

        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer.book = book  # attach modified workbook
            write_sheets(writer, benchmark_data)

    except Exception as e:
        print(f"Error opening existing Excel file: {e}")
        print("Recreating the file in write mode.")
        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
            write_sheets(writer, benchmark_data)


Error opening existing Excel file: property 'book' of 'OpenpyxlWriter' object has no setter
Recreating the file in write mode.


In [ ]:
# Comprehensive testing of all DD_benchmark files
import os
import glob
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

def test_all_dd_benchmark_files():
    """
    Iteratively test all .qasm files in the DD_benchmark directory
    with dd_sparse_numpy, dd_sparse_cupy, and cudaq_sv
    """

    # Define the base directory for benchmarks
    base_dir = Path('../ext/DD_benchmark')

    # Find all .qasm files recursively
    qasm_files = list(base_dir.glob('**/*.qasm'))

    print(f"Found {len(qasm_files)} QASM files to test")

    # Initialize results storage
    all_results = []

    # Define methods to test
    methods = ['dd_sparse_numpy', 'dd_sparse_cupy', 'cudaq_sv']

    # Test each file
    for i, qasm_file in enumerate(qasm_files):
        print(f"\n{'='*60}")
        print(f"Testing file {i+1}/{len(qasm_files)}: {qasm_file.name}")
        print(f"Path: {qasm_file}")
        print(f"{'='*60}")

        try:
            # Parse the QASM file
            NUM_QUBITS, operations = parse_qasm_to_custom(qasm_file)
            INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

            print(f"Number of qubits: {NUM_QUBITS}")
            print(f"Number of operations: {len(operations)}")

            # Initialize result for this file
            file_result = {
                'file_path': str(qasm_file),
                'file_name': qasm_file.name,
                'benchmark_category': qasm_file.parent.name,
                'num_qubits': NUM_QUBITS,
                'num_operations': len(operations),
            }

            # Test each method
            for method in methods:
                print(f"\n--- Testing with {method} ---")

                try:
                    if method == 'dd_sparse_numpy':
                        result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
                    elif method == 'dd_sparse_cupy':
                        result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='strong_simulation')
                    elif method == 'cudaq_sv':
                        # For cudaq_sv, we need the directory path, not the file path
                        circuit_dir = qasm_file.parent
                        result = cudaq_sv(str(circuit_dir), NUM_QUBITS)

                    # Store results
                    file_result[f'{method}_time'] = result.get('time_spent', None)
                    file_result[f'{method}_cpu_mem'] = result.get('peak_mem', None)
                    file_result[f'{method}_gpu_mem'] = result.get('peak_gpu_mem', None)
                    file_result[f'{method}_error'] = result.get('error', None)

                    if result.get('error') is None:
                        print(f"✓ {method} completed successfully")
                        print(f"  Time: {result.get('time_spent', 'N/A')}s")
                        if 'peak_mem' in result:
                            print(f"  CPU Memory: {result.get('peak_mem', 'N/A')}MB")
                        if 'peak_gpu_mem' in result:
                            print(f"  GPU Memory: {result.get('peak_gpu_mem', 'N/A')}MB")
                    else:
                        print(f"✗ {method} failed: {result.get('error', 'Unknown error')}")

                except Exception as e:
                    print(f"✗ {method} failed with exception: {str(e)}")
                    file_result[f'{method}_time'] = None
                    file_result[f'{method}_cpu_mem'] = None
                    file_result[f'{method}_gpu_mem'] = None
                    file_result[f'{method}_error'] = str(e)

            all_results.append(file_result)

        except Exception as e:
            print(f"✗ Failed to parse file {qasm_file}: {str(e)}")
            # Still record the failed file
            all_results.append({
                'file_path': str(qasm_file),
                'file_name': qasm_file.name,
                'benchmark_category': qasm_file.parent.name,
                'num_qubits': None,
                'num_operations': None,
                'parse_error': str(e)
            })

    return all_results

def save_comprehensive_results(results, output_file='comprehensive_benchmark_results.xlsx'):
    """
    Save the comprehensive benchmark results to an Excel file
    """
    df = pd.DataFrame(results)

    # Create Excel writer
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Save full results
        df.to_excel(writer, sheet_name='All_Results', index=False)

        # Create summary by benchmark category
        if 'benchmark_category' in df.columns:
            summary_stats = []
            for category in df['benchmark_category'].unique():
                category_df = df[df['benchmark_category'] == category]

                # Calculate average times for successful runs
                for method in ['dd_sparse_numpy', 'dd_sparse_cupy', 'cudaq_sv']:
                    time_col = f'{method}_time'
                    if time_col in category_df.columns:
                        successful_times = category_df[category_df[f'{method}_error'].isna()][time_col]
                        avg_time = successful_times.mean() if len(successful_times) > 0 else None
                        success_rate = len(successful_times) / len(category_df) * 100

                        summary_stats.append({
                            'benchmark_category': category,
                            'method': method,
                            'total_files': len(category_df),
                            'successful_runs': len(successful_times),
                            'success_rate_percent': success_rate,
                            'avg_time_seconds': avg_time
                        })

            summary_df = pd.DataFrame(summary_stats)
            summary_df.to_excel(writer, sheet_name='Summary_by_Category', index=False)

    print(f"Results saved to {output_file}")
    return df

# Function to run a subset of tests (useful for debugging or partial runs)
def test_benchmark_subset(pattern='**/VQE/*.qasm', max_files=5):
    """
    Test a subset of benchmark files matching a pattern
    """
    base_dir = Path('../ext/DD_benchmark')
    qasm_files = list(base_dir.glob(pattern))[:max_files]

    print(f"Testing {len(qasm_files)} files matching pattern: {pattern}")

    all_results = []
    methods = ['dd_sparse_numpy', 'dd_sparse_cupy', 'cudaq_sv']

    for qasm_file in qasm_files:
        print(f"\nTesting: {qasm_file.name}")

        try:
            NUM_QUBITS, operations = parse_qasm_to_custom(qasm_file)
            INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

            file_result = {
                'file_name': qasm_file.name,
                'num_qubits': NUM_QUBITS,
                'num_operations': len(operations),
            }

            for method in methods:
                try:
                    if method == 'dd_sparse_numpy':
                        result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
                    elif method == 'dd_sparse_cupy':
                        result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
                    elif method == 'cudaq_sv':
                        circuit_dir = qasm_file.parent
                        result = cudaq_sv(str(circuit_dir), NUM_QUBITS)

                    file_result[f'{method}_time'] = result.get('time_spent', None)
                    file_result[f'{method}_error'] = result.get('error', None)

                    status = "✓" if result.get('error') is None else "✗"
                    print(f"  {method}: {status} ({result.get('time_spent', 'N/A')}s)")

                except Exception as e:
                    file_result[f'{method}_time'] = None
                    file_result[f'{method}_error'] = str(e)
                    print(f"  {method}: ✗ {str(e)}")

            all_results.append(file_result)

        except Exception as e:
            print(f"  Parse error: {str(e)}")

    return all_results
# Start comprehensive testing of all DD_benchmark files
print("Starting comprehensive testing of all DD_benchmark files...")
print("This may take a considerable amount of time depending on the number of files and their complexity.")

# First, let's do a quick count of files to test
base_dir = Path('../ext/DD_benchmark')
qasm_files = list(base_dir.glob('**/*.qasm'))
print(f"Total files found: {len(qasm_files)}")

# Show distribution by directory
from collections import defaultdict
dir_counts = defaultdict(int)
for qasm_file in qasm_files:
    # Get the immediate parent directory name
    dir_counts[qasm_file.parent.name] += 1

print("\nFiles by benchmark category:")
for dirname, count in sorted(dir_counts.items()):
    print(f"  {dirname}: {count} files")

# Ask user if they want to proceed with full test or subset
print("\nOptions:")
print("1. Run full test on all files (may take hours)")
print("2. Run test on a small subset for verification")
print("3. Run test on specific benchmark category")

# For now, let's start with a small subset to verify everything works
print("\nStarting with a small subset test to verify the setup...")
subset_results = test_benchmark_subset(pattern='**/VQE/*.qasm', max_files=3)

# Display results
print("\n" + "="*60)
print("SUBSET TEST RESULTS SUMMARY")
print("="*60)
for result in subset_results:
    print(f"\nFile: {result['file_name']}")
    print(f"Qubits: {result.get('num_qubits', 'N/A')}")
    for method in ['dd_sparse_numpy', 'dd_sparse_cupy', 'cudaq_sv']:
        time_key = f'{method}_time'
        error_key = f'{method}_error'
        if time_key in result:
            if result.get(error_key) is None:
                print(f"  {method}: ✓ {result[time_key]:.4f}s")
            else:
                print(f"  {method}: ✗ {result.get(error_key, 'Unknown error')}")
# Full comprehensive test execution
# Set this to True to run the full test on all files
RUN_FULL_TEST = False

# Set this to a specific pattern to test only certain categories
# Examples:
# - '**/VQE/*.qasm' for VQE benchmarks only
# - '**/QFT/*.qasm' for QFT benchmarks only
# - '**/Shor/*.qasm' for Shor benchmarks only
# - '**/*.qasm' for all files
TEST_PATTERN = '**/VQE/*.qasm'

# Maximum number of files to test (set to None for no limit)
MAX_FILES = 10

if RUN_FULL_TEST:
    print("Running FULL comprehensive test on ALL files...")
    print("WARNING: This may take several hours to complete!")

    # Run the full test
    comprehensive_results = test_all_dd_benchmark_files()

    # Save results
    results_df = save_comprehensive_results(comprehensive_results, '../result/full_comprehensive_results.xlsx')

    print(f"\n✅ Full test completed! Results saved to full_comprehensive_results.xlsx")
    print(f"Total files tested: {len(comprehensive_results)}")

    # Show summary statistics
    successful_numpy = sum(1 for r in comprehensive_results if r.get('dd_sparse_numpy_error') is None)
    successful_cupy = sum(1 for r in comprehensive_results if r.get('dd_sparse_cupy_error') is None)
    successful_cudaq = sum(1 for r in comprehensive_results if r.get('cudaq_sv_error') is None)

    print(f"\nSuccess rates:")
    print(f"  dd_sparse_numpy: {successful_numpy}/{len(comprehensive_results)} ({successful_numpy/len(comprehensive_results)*100:.1f}%)")
    print(f"  dd_sparse_cupy: {successful_cupy}/{len(comprehensive_results)} ({successful_cupy/len(comprehensive_results)*100:.1f}%)")
    print(f"  cudaq_sv: {successful_cudaq}/{len(comprehensive_results)} ({successful_cudaq/len(comprehensive_results)*100:.1f}%)")

else:
    print(f"Running targeted test with pattern: {TEST_PATTERN}")
    if MAX_FILES:
        print(f"Limited to {MAX_FILES} files")

    # Find matching files
    base_dir = Path('../ext/DD_benchmark')
    matching_files = list(base_dir.glob(TEST_PATTERN))
    if MAX_FILES:
        matching_files = matching_files[:MAX_FILES]

    print(f"Found {len(matching_files)} matching files to test")

    if len(matching_files) > 0:
        # Run targeted test
        targeted_results = []
        methods = ['dd_sparse_numpy', 'dd_sparse_cupy', 'cudaq_sv']

        for i, qasm_file in enumerate(matching_files):
            print(f"\n{'='*50}")
            print(f"Testing file {i+1}/{len(matching_files)}: {qasm_file.name}")
            print(f"{'='*50}")

            try:
                # Parse the QASM file
                NUM_QUBITS, operations = parse_qasm_to_custom(qasm_file)
                INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

                print(f"Qubits: {NUM_QUBITS}, Operations: {len(operations)}")

                file_result = {
                    'file_path': str(qasm_file),
                    'file_name': qasm_file.name,
                    'benchmark_category': qasm_file.parent.name,
                    'num_qubits': NUM_QUBITS,
                    'num_operations': len(operations),
                }

                # Test each method
                for method in methods:
                    print(f"\n--- Testing {method} ---")

                    try:
                        start_time = time.time()

                        if method == 'dd_sparse_numpy':
                            result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
                        elif method == 'dd_sparse_cupy':
                            result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS, MODE='weak_simulation')
                        elif method == 'cudaq_sv':
                            circuit_dir = qasm_file.parent
                            result = cudaq_sv(str(circuit_dir), NUM_QUBITS)

                        # Store results
                        file_result[f'{method}_time'] = result.get('time_spent', None)
                        file_result[f'{method}_cpu_mem'] = result.get('peak_mem', None)
                        file_result[f'{method}_gpu_mem'] = result.get('peak_gpu_mem', None)
                        file_result[f'{method}_error'] = result.get('error', None)

                        if result.get('error') is None:
                            print(f"✅ Success - Time: {result.get('time_spent', 'N/A'):.4f}s")
                            if 'peak_mem' in result and result['peak_mem'] is not None:
                                print(f"   CPU Memory: {result['peak_mem']:.2f}MB")
                            if 'peak_gpu_mem' in result and result['peak_gpu_mem'] is not None:
                                print(f"   GPU Memory: {result['peak_gpu_mem']:.2f}MB")
                        else:
                            print(f"❌ Failed: {result.get('error', 'Unknown error')}")

                    except Exception as e:
                        print(f"❌ Exception: {str(e)}")
                        file_result[f'{method}_time'] = None
                        file_result[f'{method}_cpu_mem'] = None
                        file_result[f'{method}_gpu_mem'] = None
                        file_result[f'{method}_error'] = str(e)

                targeted_results.append(file_result)

            except Exception as e:
                print(f"❌ Parse error: {str(e)}")
                targeted_results.append({
                    'file_path': str(qasm_file),
                    'file_name': qasm_file.name,
                    'benchmark_category': qasm_file.parent.name,
                    'parse_error': str(e)
                })

        # Save targeted results
        if targeted_results:
            results_df = save_comprehensive_results(targeted_results, f'targeted_results_{TEST_PATTERN.replace("**/", "").replace("/*.qasm", "")}.xlsx')
            print(f"\n✅ Targeted test completed! Results saved.")

            # Show summary
            total_tests = len(targeted_results)
            successful_numpy = sum(1 for r in targeted_results if r.get('dd_sparse_numpy_error') is None)
            successful_cupy = sum(1 for r in targeted_results if r.get('dd_sparse_cupy_error') is None)
            successful_cudaq = sum(1 for r in targeted_results if r.get('cudaq_sv_error') is None)

            print(f"\nSUMMARY:")
            print(f"Total files tested: {total_tests}")
            print(f"dd_sparse_numpy success: {successful_numpy}/{total_tests} ({successful_numpy/total_tests*100:.1f}%)")
            print(f"dd_sparse_cupy success: {successful_cupy}/{total_tests} ({successful_cupy/total_tests*100:.1f}%)")
            print(f"cudaq_sv success: {successful_cudaq}/{total_tests} ({successful_cudaq/total_tests*100:.1f}%)")
    else:
        print("No files found matching the pattern!")

Starting comprehensive testing of all DD_benchmark files...
This may take a considerable amount of time depending on the number of files and their complexity.
Total files found: 419

Files by benchmark category:
  DJ: 129 files
  QAOA: 14 files
  QFT: 129 files
  QNN: 129 files
  Shor: 2 files
  VQE: 14 files
  renamed_shor: 2 files

Options:
1. Run full test on all files (may take hours)
2. Run test on a small subset for verification
3. Run test on specific benchmark category

Starting with a small subset test to verify the setup...
Testing 3 files matching pattern: **/VQE/*.qasm

Testing: vqe_indep_qiskit_10.qasm
L_U.shape:  (21, 21)
L_up.shape:  (21, 21)
L_up_target.shape:  (21, 21)
L_up_control.shape:  (21, 21)
R_U.shape:  (21, 21)
R_up.shape:  (21, 21)
R_up_target.shape:  (21, 21)
L_U PASS
R_U PASS
L_U.shape:  (23, 23)
L_up.shape:  (21, 21)
L_up_target.shape:  (21, 21)
L_up_control.shape:  (21, 21)
R_U.shape:  (23, 23)
R_up.shape:  (21, 21)
R_up_target.shape:  (21, 21)
L_U PASS
R_

/mnt/c/Users/Yu-Hsuan Wu/DD-Matrix/.venv/lib/python3.12/site-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


  dd_sparse_numpy: ✓ (11.455506801605225s)
L_U.shape:  (21, 21)  max:  20
L_up.shape:  (21, 21)  max:  16
L_up_target.shape:  (21, 21)  max:  18
L_up_control.shape:  (21, 21)  max:  16
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (21, 21)>
  Coords	Values
  (17, 15)	(0.7754258195280138+0j)
  (18, 15)	(0.6314386735141492+0j)
  (18, 16)	(1.406864493042163+0j)
R_U.shape:  (21, 21)  max:  18
R_up.shape:  (21, 21)  max:  16
R_up_target.shape:  (21, 21)  max:  18
L_U PASS
R_U PASS


/mnt/c/Users/Yu-Hsuan Wu/DD-Matrix/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


L_U.shape:  (23, 23)  max:  20
L_up.shape:  (21, 21)  max:  14
L_up_target.shape:  (21, 21)  max:  16
L_up_control.shape:  (21, 21)  max:  14
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (21, 21)>
  Coords	Values
  (15, 13)	(0.04349907106002518+0j)
  (16, 13)	(-0.99905346744652+0j)
  (16, 14)	(-0.9555543963864949+0j)
R_U.shape:  (23, 23)  max:  22
R_up.shape:  (21, 21)  max:  14
R_up_target.shape:  (21, 21)  max:  16
L_U PASS
R_U PASS
L_U.shape:  (25, 25)  max:  20
L_up.shape:  (19, 19)  max:  12
L_up_target.shape:  (19, 19)  max:  14
L_up_control.shape:  (19, 19)  max:  12
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (19, 19)>
  Coords	Values
  (13, 11)	(0.08063317816879827+0j)
  (14, 11)	(-0.9967438440132945+0j)
  (14, 12)	(-0.9161106658444963+0j)
R_U.shape:  (25, 25)  max:  24
R_up.shape:  (19, 19)  max:  12
R_up_target.shape:  (19, 19)  max:  14
L_U PASS
R_U

...

Traceback (most recent call last):
  File "/tmp/ipykernel_878/2307632313.py", line 31, in dd_sparse_numpy
    exec_time, sim_result, sim_time = dd_sparse_cpu(operations, INIT_MATRIX_SIZE, NUM_QUBITS, TYPE='numpy', START_TIME=np_start, MODE=MODE)
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/2841208764.py", line 445, in dd_sparse_cpu
    sim_result, sim_time = weak_simulation(L, R, Mapping.toarray(), NUM_QUBITS, TYPE, START_TIME, num_shots=1024)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/731406547.py", line 571, in weak_simulation
    return weak_sim.get_result(), time.time() - start_time
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/731406547.py", line 551, in get_result
    self.simulation()
  File "/tmp/ipykernel_878/731406547.py", line 537, in simulation

  dd_sparse_numpy: ✗ (60.361392974853516s)
L_U.shape:  (25, 25)  max:  24
L_up.shape:  (25, 25)  max:  20
L_up_target.shape:  (25, 25)  max:  22
L_up_control.shape:  (25, 25)  max:  20
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (25, 25)>
  Coords	Values
  (21, 19)	(0.706181922602642+0j)
  (22, 19)	(0.7080304316830146+0j)
  (22, 20)	(1.4142123542856566+0j)
R_U.shape:  (25, 25)  max:  22
R_up.shape:  (25, 25)  max:  20
R_up_target.shape:  (25, 25)  max:  22
L_U PASS
R_U PASS
L_U.shape:  (27, 27)  max:  24
L_up.shape:  (25, 25)  max:  18
L_up_target.shape:  (25, 25)  max:  20
L_up_control.shape:  (25, 25)  max:  18
L_up_control:  <Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 3 stored elements and shape (25, 25)>
  Coords	Values
  (19, 17)	(0.9999999514291196+0j)
  (20, 17)	(-0.000311675726447261+0j)
  (20, 18)	(0.9996882757026723+0j)
R_U.shape:  (27, 27)  max:  26
R_up.shape:  (25, 25)  max:  18
R_up_targe

Traceback (most recent call last):
  File "/tmp/ipykernel_878/1264509649.py", line 57, in dd_sparse_cupy
    exec_time, sim_result, sim_time = dd_sparse_cuda(operations, INIT_MATRIX_SIZE, NUM_QUBITS, TYPE='cupy', START_TIME=cp_start, MODE=MODE, DEBUG=DEBUG)
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/1602655269.py", line 447, in dd_sparse_cuda
    sim_result, sim_time = weak_simulation(L, R, Mapping.toarray(), NUM_QUBITS, TYPE, START_TIME, num_shots=1024)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/731406547.py", line 571, in weak_simulation
    return weak_sim.get_result(), time.time() - start_time
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_878/731406547.py", line 551, in get_result
    self.simulation()
  File "/tmp/ipykernel_878/731406547.py

  dd_sparse_cupy: ✗ (75.12353682518005s)
  cudaq_sv: ✗ [Errno 2] No such file or directory: '../ext/DD_benchmark/MQTbench/VQEcircuit.qasm'

SUBSET TEST RESULTS SUMMARY

File: vqe_indep_qiskit_10.qasm
Qubits: 10
  dd_sparse_numpy: ✓ 11.4555s
  dd_sparse_cupy: ✓ 24.8111s
  cudaq_sv: ✗ [Errno 2] No such file or directory: '../ext/DD_benchmark/MQTbench/VQEcircuit.qasm'

File: vqe_indep_qiskit_11.qasm
Qubits: 11
  dd_sparse_numpy: ✗ Timeout during simulation: 60s.
  dd_sparse_cupy: ✗ Timeout during simulation: 60s.
  cudaq_sv: ✗ [Errno 2] No such file or directory: '../ext/DD_benchmark/MQTbench/VQEcircuit.qasm'

File: vqe_indep_qiskit_12.qasm
Qubits: 12
  dd_sparse_numpy: ✗ Timeout during simulation: 60s.
  dd_sparse_cupy: ✗ Timeout during simulation: 60s.
  cudaq_sv: ✗ [Errno 2] No such file or directory: '../ext/DD_benchmark/MQTbench/VQEcircuit.qasm'
Running targeted test with pattern: **/VQE/*.qasm
Limited to 10 files
Found 10 matching files to test

Testing file 1/10: vqe_indep_qiskit_1

KeyboardInterrupt: 

In [ ]:
# from typing import Dict, Tuple
def cudaq_result_to_probs(cudaq_result: str, sample_size: int) -> Dict[str, float]:
    """
    Convert cudaQ result string to a dictionary of probabilities.
    """
    cudaq_result = cudaq_result.strip().lstrip("{").rstrip("}").strip()
    cudaq_probs = {}
    if cudaq_result:
        for pair in cudaq_result.split():
            if ":" not in pair:
                continue
            state, prob_str = pair.split(":")
            cudaq_probs[state] = float(prob_str) / sample_size
    return cudaq_probs

def dd_result_to_probs(dd_result: Dict[Tuple[int, ...], complex]) -> Dict[str, float]:
    """
    Convert dd_sparse result dictionary to a dictionary of probabilities.
    """
    return {
        "".join(map(str, state)): abs(amp) ** 2
        for state, amp in dd_result.items()
    }

def dd_and_cudaq_is_equal(
    my_result: Dict[Tuple[int, ...], complex],
    cudaq_result: str,
    sample_size: int = 1024,
    tol: float = 1e-6
) -> bool:
    """
    Compare probabilities from a DD-sparse simulator and a cudaQ simulator.

    Parameters
    ----------
    my_result : dict
        Key: tuple of 0/1 ints representing a basis state.
        Value: complex amplitude from dd_sparse.
    cudaq_result : str
        cudaQ output string, e.g. "{ 11:500 00:500 }".
        Each probability is scaled by 1000.
    tol : float, optional
        Allowed absolute difference between probabilities.

    Returns
    -------
    bool
        True if every state's probabilities match within `tol`, else False.
    """

    cudaq_probs = cudaq_result_to_probs(cudaq_result, sample_size)
    dd_probs = dd_result_to_probs(my_result)

    all_states = set(dd_probs) | set(cudaq_probs)

    for st in all_states:
        p_dd = dd_probs.get(st, 0.0)
        p_cq = cudaq_probs.get(st, 0.0)
        if abs(p_dd - p_cq) > tol:
            return False

    return True


In [ ]:
# import random

def dd_sample_result_initializer() -> Dict[str, int]:
    """
    Initialize the sample result dictionary with all possible states.
    """
    dd_sample_result = {}

    for i in range(2 ** NUM_QUBITS):
        state = bin(i)[2:].zfill(NUM_QUBITS)
        dd_sample_result[state] = 0

    return dd_sample_result

def dd_sampler(dd_result: Dict[Tuple[int, ...], complex], sample_size: int) -> Dict[str, float]:
    """
    Sample from the DD result dictionary to get a sample distribution.
    Parameters
    ----------
    dd_result : dict
        Key: tuple of 0/1 ints representing a basis state.
        Value: complex amplitude from dd_sparse.
    sample_size : int
        Number of samples to draw from the distribution.
    Returns
    -------
    dict
        Key: tuple of 0/1 ints representing a basis state.
        Value: float representing the probability of that state in the sample.
    """

    dd_probs = dd_result_to_probs(dd_result)
    dd_sample_result = dd_sample_result_initializer()

    sampled_states = random.choices(list(dd_probs.keys()), weights = list(dd_probs.values()), k = sample_size)

    for state in sampled_states:
        dd_sample_result[state] += 1

    for state in dd_sample_result:
        dd_sample_result[state] /= sample_size

    return dd_sample_result


In [ ]:
import os
from pathlib import Path
import pandas as pd
from typing import List, Dict, Tuple
import pandas as pd
import openpyxl

def find_all_benchmark_files() -> Dict[str, List[Path]]:
    """
    Find all benchmark files in the specified directories:
    - .qasm files in MQTbench
    - .real files in Revlib_renamed
    - .qasm files in AutoQ/benchmark/all/ (if exists)

    Returns:
        Dictionary with file types as keys and lists of Path objects as values
    """
    base_path = Path("../ext/DD_benchmark")

    files_found = {
        'mqt_qasm': [],
        'revlib_real': [],
        'autoq_qasm': []
    }

    # Find MQTbench .qasm files
    mqt_path = base_path / "MQTbench"
    if mqt_path.exists():
        for qasm_file in mqt_path.rglob("*.qasm"):
            files_found['mqt_qasm'].append(qasm_file)

    # Find Revlib .real files
    revlib_path = base_path / "Revlib_renamed"
    if revlib_path.exists():
        for real_file in revlib_path.rglob("*.real"):
            files_found['revlib_real'].append(real_file)

    # Find AutoQ benchmark .qasm files
    autoq_path = Path("../ext/AutoQ/benchmarks/all")
    if autoq_path.exists():
        for qasm_file in autoq_path.rglob("*.qasm"):
            files_found['autoq_qasm'].append(qasm_file)
    else:
        print("AutoQ benchmark directory does not exist. Skipping AutoQ files search.")
    return files_found

def iterate_all_benchmark_files():
    """
    Iterate through all benchmark files and provide summary statistics
    """
    print("🔍 Searching for benchmark files...")

    files_found = find_all_benchmark_files()

    # Print summary
    total_files = sum(len(file_list) for file_list in files_found.values())
    print(f"\n📊 Summary of files found:")
    print(f"  MQTbench (.qasm): {len(files_found['mqt_qasm'])} files")
    print(f"  Revlib (.real):   {len(files_found['revlib_real'])} files")
    print(f"  AutoQ (.qasm):    {len(files_found['autoq_qasm'])} files")
    print(f"  Total files:      {total_files} files")

    return files_found

# Run the iterator

benchmark_files = iterate_all_benchmark_files()
benchmark_data = {}
for file in benchmark_files['mqt_qasm'] + benchmark_files['revlib_real'] + benchmark_files['autoq_qasm']:
    methods = ['NumPy + PySparse', 'CuPy + cuSparse', 'CudaQ (State Vector)']
    subfields = ['Time (s)', 'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error']

    columns = [('Qubits', '')]
    rows = []

    for method in methods:
        for sub in subfields:
            if method == 'NumPy + PySparse' and sub == 'GPU Memory Usage (MB)':
                continue
            columns.append((method, sub))

    multi_columns = pd.MultiIndex.from_tuples(columns)

    flag = (True, True, True)

    NUM_QUBITS, operations = parse_qasm_to_custom(file)

    INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1
    print(f"Qubit count: {NUM_QUBITS}")
        # print("Operations:", operations)

    if flag[0]:
        np_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
    if flag[1]:
        cp_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
    if flag[2]:
        cudaq_result = cudaq_sv(path, NUM_QUBITS)

    rows.append([NUM_QUBITS,
                np_result['time_spent'] if flag[0] else '--',
                np_result['peak_mem'] if flag[0] else '--',
                np_result['error'] if flag[0] else '--',
                cp_result['time_spent'] if flag[1] else '--',
                cp_result['peak_mem'] if flag[1] else '--',
                cp_result['peak_gpu_mem'] if flag[1] else '--',
                cp_result['error'] if flag[1] else '--',
                cudaq_result['time_spent'] if flag[2] else '--',
                cudaq_result['peak_mem'] if flag[2] else '--',
                cudaq_result['peak_gpu_mem'] if flag[2] else '--',
                cudaq_result['error'] if flag[2] else '--'
                ])


    if not np_result['error'] is None:
        flag = (False, flag[1], flag[2])

    if not cp_result['error'] is None:
        flag = (flag[0], False, flag[2])

    if not cudaq_result['error'] is None:
        flag = (flag[0], flag[1], False)


    benchmark_data[file] = rows